# Machine Learning Workflow


## Imports


In [ ]:
from __future__ import annotations
import errno
import logging
import os
import re
import sys
import tempfile
import warnings
from contextlib import contextmanager
from datetime import datetime, timezone, timedelta
from functools import partial
from math import pi
from pathlib import Path
from typing import Any
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import polars as pl
import psycopg2
import yaml
from deltalake import DeltaTable, write_deltalake
from feast import Entity, FeatureService, FeatureView, Field, FeatureStore
from feast.infra.offline_stores.contrib.postgres_offline_store.postgres_source import PostgreSQLSource
from feast.types import Float64, Int64
from psycopg2.extras import execute_values
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from utilsforecast.plotting import plot_series
from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import optuna
from mlforecast.auto import (
    AutoCatboost,
    AutoLightGBM,
    AutoModel,
    AutoRandomForest,
    AutoRidge,
    AutoXGBoost,
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
import statsforecast.models as stats_models
from statsforecast.utils import ConformalIntervals
import pickle
from statsforecast import StatsForecast
from datetime import timedelta
from feast import Entity, FeatureService, FeatureView, Field
from feast.infra.offline_stores.contrib.postgres_offline_store.postgres_source import (
    PostgreSQLSource,
)
from datetime import datetime, timezone
from feast import FeatureStore
from kedro.pipeline import node


## Configuration


In [ ]:
# config_resolvers.py
def configured_list(mapping: dict[str, Any], key: str) -> list[str]:
    return list(mapping.get(key, []))


In [ ]:
# config_resolvers.py
def resolve_column_config(columns: dict[str, Any]) -> dict[str, list[str]]:
    entity_columns = configured_list(columns, "entity")
    price_columns = configured_list(columns, "price")
    analytics_calendar_columns = configured_list(columns, "analytics_calendar")
    target_columns = configured_list(columns, "target")
    tier_1_feature_columns = configured_list(columns, "tier_1_features")
    model_time_feature_columns = configured_list(columns, "model_time_features")
    indicator_columns = configured_list(columns, "indicator")
    condition_columns = configured_list(columns, "condition")
    strategy_label_columns = configured_list(columns, "strategy_label")
    output_audit_columns = configured_list(columns, "output_audit")
    close_model_time_feature_columns = configured_list(
        columns,
        "close_model_time_features",
    )
    tier_2_feature_columns = list(
        columns.get(
            "tier_2_features",
            [*tier_1_feature_columns, *model_time_feature_columns],
        )
    )
    tier_3_feature_columns = list(
        columns.get(
            "tier_3_features",
            tier_2_feature_columns,
        )
    )
    indicator_feature_columns = list(
        columns.get(
            "indicator_features",
            [
                *entity_columns,
                *output_audit_columns,
                *price_columns,
                *analytics_calendar_columns,
                *target_columns,
                *tier_1_feature_columns,
                *model_time_feature_columns,
                *indicator_columns,
            ],
        )
    )
    conventional_gap_trading_columns = list(
        columns.get(
            "conventional_gap_trading",
            [
                *indicator_feature_columns,
                *condition_columns,
                *strategy_label_columns,
            ],
        )
    )
    close_model_dataset_columns = list(
        columns.get(
            "close_model_dataset",
            [
                "unique_id",
                "ds",
                "y",
                *close_model_time_feature_columns,
                *output_audit_columns,
            ],
        )
    )
    return {
        "entity": entity_columns,
        "price": price_columns,
        "analytics_calendar": analytics_calendar_columns,
        "target": target_columns,
        "tier_1_features": tier_1_feature_columns,
        "tier_2_features": tier_2_feature_columns,
        "tier_3_features": tier_3_feature_columns,
        "fourier_time_encoding": configured_list(columns, "fourier_time_encoding"),
        "model_time_features": model_time_feature_columns,
        "indicator": indicator_columns,
        "condition": condition_columns,
        "strategy_required_conditions": configured_list(
            columns,
            "strategy_required_conditions",
        ),
        "strategy_label": strategy_label_columns,
        "output_audit": output_audit_columns,
        "indicator_features": indicator_feature_columns,
        "conventional_gap_trading": conventional_gap_trading_columns,
        "close_model_time_features": close_model_time_feature_columns,
        "close_model_dataset": close_model_dataset_columns,
        "model_ready": list(
            columns.get(
                "model_ready",
                [
                    *tier_1_feature_columns,
                    *model_time_feature_columns,
                    *indicator_columns,
                ],
            )
        ),
    }


## Feature Sets


In [ ]:
# features/feature_sets.py; notebook-safe path resolver
def _kedro_project_dir() -> Path:
    env_path = os.getenv("KEDRO_PROJECT_DIR")
    if env_path:
        return Path(env_path).expanduser().resolve()

    kedro_project = globals().get("KEDRO_PROJECT")
    if kedro_project:
        return Path(kedro_project).expanduser().resolve()

    workspace_root = globals().get("WORKSPACE_PROJECT_ROOT")
    candidates = [
        Path("/opt/kedro_project"),
        Path(workspace_root) / "kedro_project" if workspace_root else None,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / "kedro_project",
        Path.cwd().parent / "kedro_project",
    ]
    for candidate in candidates:
        if candidate is None:
            continue
        candidate = Path(candidate).expanduser().resolve()
        if (candidate / "conf" / "base" / "parameters_data_preprocessing.yml").exists():
            return candidate
    return Path.cwd().resolve()


In [ ]:
# features/feature_sets.py
def _load_data_preprocessing_columns() -> dict[str, list[str]]:
    parameters_path = (
        _kedro_project_dir() / "conf" / "base" / "parameters_data_preprocessing.yml"
    )
    parameters = yaml.safe_load(parameters_path.read_text()) or {}
    data_preprocessing = parameters.get("stock_close_data_preprocessing", {})
    return resolve_column_config(data_preprocessing.get("columns", {}))


In [ ]:
# features/feature_sets.py
def _column_config() -> dict[str, list[str]]:
    return _load_data_preprocessing_columns()


## Runtime Helpers


In [ ]:
# ml/runtime.py
def cpu_count_from_env(env_name: str, default: int = 1) -> int:
    requested = int(os.getenv(env_name, str(default)))
    if requested <= 0:
        return os.cpu_count() or 1
    return requested


In [ ]:
# ml/runtime.py
def bool_from_env(env_name: str, default: bool = False) -> bool:
    value = os.getenv(env_name)
    if value is None:
        return default
    return value.lower() in {"1", "true", "yes"}


In [ ]:
# ml/runtime.py
def filter_sklearn_parallel_warnings() -> None:
    warnings.filterwarnings(
        "ignore",
        message=(
            "`sklearn.utils.parallel.delayed` should be used with "
            "`sklearn.utils.parallel.Parallel`.*"
        ),
        category=UserWarning,
        module="sklearn.utils.parallel",
    )


## Metrics


In [ ]:
# ml/metrics.py
def model_prediction_columns(df: pd.DataFrame) -> list[str]:
    return [
        column
        for column in df.columns
        if column not in BASE_COLUMNS
        and "-lo-" not in column
        and "-hi-" not in column
    ]


In [ ]:
# ml/metrics.py
def add_previous_actual_close(
    joined_df: pd.DataFrame,
    train_df: pd.DataFrame,
) -> pd.DataFrame:
    last_train_close = (
        train_df.sort_values(["unique_id", "ds"])
        .groupby("unique_id", observed=True)["y"]
        .last()
    )

    ordered = joined_df.sort_values(["unique_id", "ds"]).copy()
    previous_close = ordered.groupby("unique_id", observed=True)["y"].shift(1)
    ordered["previous_actual_close"] = previous_close.fillna(
        ordered["unique_id"].map(last_train_close)
    )
    return ordered


In [ ]:
# ml/metrics.py
def build_long_direction_frame(
    joined_df: pd.DataFrame,
    train_df: pd.DataFrame,
) -> pd.DataFrame:
    direction_df = add_previous_actual_close(joined_df, train_df)
    models = model_prediction_columns(direction_df)
    direction_df["actual_long"] = (
        direction_df["y"] > direction_df["previous_actual_close"]
    )

    for model in models:
        direction_df[f"{model}_long"] = (
            direction_df[model] > direction_df["previous_actual_close"]
        )

    return direction_df[
        [
            "unique_id",
            "ds",
            "y",
            "previous_actual_close",
            "actual_long",
            *[f"{model}_long" for model in models],
        ]
    ]


In [ ]:
# ml/metrics.py
def long_only_directional_metrics(
    joined_df: pd.DataFrame,
    train_df: pd.DataFrame,
) -> pd.DataFrame:
    direction_df = build_long_direction_frame(joined_df, train_df)
    rows = []

    for column in direction_df.columns:
        if not column.endswith("_long") or column == "actual_long":
            continue

        model = column.removesuffix("_long")
        valid_rows = direction_df[["actual_long", column]].dropna()

        if valid_rows.empty:
            rows.append(
                {
                    "model": model,
                    "long_accuracy": None,
                    "long_precision": None,
                    "long_recall": None,
                    "long_signal_rate": None,
                    "rows": 0,
                }
            )
            continue

        rows.append(
            {
                "model": model,
                "long_accuracy": accuracy_score(
                    valid_rows["actual_long"],
                    valid_rows[column],
                ),
                "long_precision": precision_score(
                    valid_rows["actual_long"],
                    valid_rows[column],
                    zero_division=0,
                ),
                "long_recall": recall_score(
                    valid_rows["actual_long"],
                    valid_rows[column],
                    zero_division=0,
                ),
                "long_signal_rate": float(valid_rows[column].mean()),
                "rows": len(valid_rows),
            }
        )

    return pd.DataFrame(rows)


## Plotting


In [ ]:
# ml/plots.py
def forecast_model_columns(df: pd.DataFrame) -> list[str]:
    return [
        column
        for column in df.columns
        if column not in {"unique_id", "ds", "y"}
        and "-lo-" not in column
        and "-hi-" not in column
    ]


In [ ]:
# ml/plots.py
def _safe_name(value: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", value)


In [ ]:
# ml/plots.py
def log_forecast_plots(
    *,
    train_df: pd.DataFrame,
    joined_df: pd.DataFrame,
    levels: list[int] | None = None,
    artifact_prefix: str = "plots",
) -> None:
    levels = levels or [80, 95]
    if joined_df.empty or "y" not in joined_df.columns:
        return

    train_history = train_df[["unique_id", "ds", "y"]].copy()
    test_actuals = joined_df[["unique_id", "ds", "y"]].dropna(subset=["y"]).copy()
    history_df = (
        pd.concat([train_history, test_actuals], ignore_index=True)
        .sort_values(["unique_id", "ds"])
        .drop_duplicates(subset=["unique_id", "ds"], keep="last")
    )
    plot_df = (
        joined_df.sort_values(["unique_id", "ds"])
        .drop_duplicates(subset=["unique_id", "ds"], keep="last")
        .copy()
    )

    last_train_values = (
        train_df.sort_values(["unique_id", "ds"])
        .groupby("unique_id", observed=True)["y"]
        .last()
    )

    for model_name in forecast_model_columns(plot_df):
        safe_model_name = _safe_name(model_name)
        available_levels = [
            level
            for level in levels
            if {
                f"{model_name}-lo-{level}",
                f"{model_name}-hi-{level}",
            }.issubset(plot_df.columns)
        ]
        interval_columns = [
            f"{model_name}-{bound}-{level}"
            for level in available_levels
            for bound in ("lo", "hi")
        ]

        figure = plot_series(
            df=history_df,
            forecasts_df=plot_df[["unique_id", "ds", model_name, *interval_columns]],
            models=[model_name],
            level=available_levels or None,
            max_insample_length=120,
            plot_random=False,
            engine="matplotlib",
        )
        mlflow.log_figure(
            figure=figure,
            artifact_file=f"{artifact_prefix}/forecasts/{safe_model_name}_forecast.png",
        )
        plt.close(figure)

        _log_directional_confusion_matrix(
            plot_df=plot_df,
            model_name=model_name,
            safe_model_name=safe_model_name,
            last_train_values=last_train_values,
            artifact_prefix=artifact_prefix,
        )


In [ ]:
# ml/plots.py
def _log_directional_confusion_matrix(
    *,
    plot_df: pd.DataFrame,
    model_name: str,
    safe_model_name: str,
    last_train_values: pd.Series,
    artifact_prefix: str,
) -> None:
    direction_df = (
        plot_df[["unique_id", "ds", "y", model_name]]
        .dropna(subset=["y", model_name])
        .sort_values(["unique_id", "ds"])
        .copy()
    )
    if direction_df.empty:
        return

    previous_actual = direction_df.groupby("unique_id", observed=True)["y"].shift(1)
    train_baseline = direction_df["unique_id"].map(last_train_values)
    previous_actual = previous_actual.fillna(train_baseline)

    valid_mask = previous_actual.notna()
    if not valid_mask.any():
        return

    actual_long = direction_df.loc[valid_mask, "y"] > previous_actual.loc[valid_mask]
    predicted_long = (
        direction_df.loc[valid_mask, model_name] > previous_actual.loc[valid_mask]
    )

    long_accuracy = accuracy_score(actual_long, predicted_long)
    figure, axis = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_true=actual_long,
        y_pred=predicted_long,
        labels=[False, True],
        display_labels=["not_long", "long"],
        cmap="Blues",
        values_format="d",
        colorbar=False,
        ax=axis,
    )
    axis.set_title(
        f"{model_name} - Long Direction Confusion Matrix\n"
        f"Long Accuracy: {long_accuracy:.4f}"
    )
    figure.tight_layout()
    mlflow.log_figure(
        figure=figure,
        artifact_file=(
            f"{artifact_prefix}/confusion_matrices/"
            f"{safe_model_name}_long_confusion_matrix.png"
        ),
    )
    plt.close(figure)

    report = classification_report(
        y_true=actual_long,
        y_pred=predicted_long,
        labels=[False, True],
        target_names=["not_long", "long"],
        zero_division=0,
        output_dict=True,
    )
    report["long_accuracy"] = float(long_accuracy)
    report["observation_count"] = int(len(actual_long))
    mlflow.log_dict(
        report,
        artifact_file=(
            f"{artifact_prefix}/reports/{safe_model_name}_long_report.json"
        ),
    )


In [ ]:
# ml/plots.py
def pecnet_prediction_figure(
    *,
    predictions: np.ndarray,
    actual: np.ndarray,
    dates: pd.Series,
    model_name: str,
) -> tuple[plt.Figure, pd.DataFrame]:
    predictions = np.asarray(predictions, dtype=float).reshape(-1)
    actual = np.asarray(actual, dtype=float).reshape(-1)
    dates = pd.Series(dates).reset_index(drop=True)

    usable_length = min(len(predictions), len(actual), len(dates))
    predictions = predictions[-usable_length:]
    actual = actual[-usable_length:]
    dates = dates.iloc[-usable_length:].reset_index(drop=True)

    valid_mask = np.isfinite(actual) & np.isfinite(predictions)
    predictions = predictions[valid_mask]
    actual = actual[valid_mask]
    dates = dates[valid_mask].reset_index(drop=True)

    residuals = actual - predictions
    mae = float(np.mean(np.abs(residuals))) if len(residuals) else np.nan
    rmse = float(np.sqrt(np.mean(residuals**2))) if len(residuals) else np.nan
    long_accuracy = (
        float(np.mean((np.diff(actual) > 0) == (np.diff(predictions) > 0)))
        if len(actual) > 1
        else np.nan
    )

    comparison_df = pd.DataFrame(
        {
            "ds": dates,
            "actual": actual,
            "prediction": predictions,
            "residual": residuals,
        }
    )

    figure, axes = plt.subplots(
        nrows=2,
        ncols=2,
        figsize=(18, 11),
        constrained_layout=True,
    )
    axes[0, 0].plot(dates, actual, color="black", linewidth=2, label="Actual")
    axes[0, 0].plot(
        dates,
        predictions,
        color="darkorange",
        linewidth=2,
        label=model_name,
    )
    axes[0, 0].set_title(
        f"{model_name}: Actual vs Prediction\n"
        f"MAE={mae:,.4f} | RMSE={rmse:,.4f} | Long Accuracy={long_accuracy:.2%}"
    )
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.25)

    axes[0, 1].scatter(actual, predictions, alpha=0.7, color="steelblue")
    if len(actual):
        value_min = min(actual.min(), predictions.min())
        value_max = max(actual.max(), predictions.max())
        axes[0, 1].plot(
            [value_min, value_max],
            [value_min, value_max],
            linestyle="--",
            color="red",
        )
    axes[0, 1].set_title("Actual vs Prediction Scatter")
    axes[0, 1].grid(alpha=0.25)

    axes[1, 0].plot(dates, residuals, color="crimson")
    axes[1, 0].axhline(0, color="black", linestyle="--")
    axes[1, 0].fill_between(dates, residuals, 0, color="crimson", alpha=0.2)
    axes[1, 0].set_title("Residuals: Actual - Prediction")
    axes[1, 0].grid(alpha=0.25)

    axes[1, 1].hist(residuals, bins=min(30, max(5, len(residuals))), color="slateblue")
    axes[1, 1].set_title("Residual Distribution")
    axes[1, 1].grid(alpha=0.25)

    return figure, comparison_df


## Common Training Utilities


In [ ]:
# ml/common.py
def model_id_columns() -> list[str]:
    return ["unique_id", "ds", "y"]


In [ ]:
# ml/common.py
def non_feature_columns() -> set[str]:
    return {
        "unique_id",
        "ds",
        "y",
        "symbol",
        "date",
        "close",
        "created_timestamp",
    }


In [ ]:
# ml/common.py
def tier_experiment_name(tier_name: str) -> str:
    prefix = os.getenv(
        "MLFLOW_TIER_EXPERIMENT_PREFIX",
        DEFAULT_MLFLOW_TIER_EXPERIMENT_PREFIX,
    )
    return f"{prefix}_{tier_name}"


In [ ]:
# ml/common.py
def configure_mlflow_tracking(
    tracking_uri: str | None = None,
    experiment_name: str | None = None,
) -> None:
    tracking_uri = tracking_uri or os.getenv(
        "MLFLOW_TRACKING_URI",
        DEFAULT_MLFLOW_TRACKING_URI,
    )
    experiment_name = experiment_name or os.getenv(
        "MLFLOW_EXPERIMENT_NAME",
        DEFAULT_MLFLOW_EXPERIMENT_NAME,
    )

    os.environ.setdefault(
        "MLFLOW_HTTP_REQUEST_TIMEOUT",
        DEFAULT_MLFLOW_REQUEST_TIMEOUT,
    )
    os.environ.setdefault(
        "MLFLOW_HTTP_REQUEST_MAX_RETRIES",
        DEFAULT_MLFLOW_REQUEST_MAX_RETRIES,
    )
    os.environ.setdefault(
        "MLFLOW_HTTP_REQUEST_BACKOFF_FACTOR",
        DEFAULT_MLFLOW_REQUEST_BACKOFF_FACTOR,
    )

    mlflow.set_tracking_uri(tracking_uri)

    try:
        mlflow.set_experiment(experiment_name)
    except Exception as exc:
        client = MlflowClient(tracking_uri=tracking_uri)
        deleted_experiment = next(
            (
                experiment
                for experiment in client.search_experiments(
                    view_type=ViewType.DELETED_ONLY,
                )
                if experiment.name == experiment_name
            ),
            None,
        )
        if deleted_experiment is not None:
            client.restore_experiment(deleted_experiment.experiment_id)
            mlflow.set_experiment(experiment_name)
            return

        raise RuntimeError(
            "MLflow experiment setup failed from Kedro. "
            f"tracking_uri={tracking_uri!r} experiment={experiment_name!r}. "
            "Check the original MLflow exception below and the mlflow container logs."
        ) from exc


In [ ]:
# ml/common.py
def split_train_test_by_horizon(
    df: pd.DataFrame,
    test_horizon: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    ordered = df.sort_values(["unique_id", "ds"]).reset_index(drop=True)
    test_index = ordered.groupby("unique_id", observed=True).tail(test_horizon).index
    test_df = ordered.loc[test_index].copy()
    train_df = ordered.drop(test_index).copy()
    return train_df, test_df


In [ ]:
# ml/common.py
def validation_reference_frame(
    train_df: pd.DataFrame,
    *,
    validation_horizon: int,
    n_windows: int,
) -> pd.DataFrame:
    row_count = max(0, int(validation_horizon)) * max(0, int(n_windows))
    if row_count == 0 or train_df.empty:
        return train_df.iloc[0:0].copy()

    return (
        train_df.sort_values(["unique_id", "ds"])
        .groupby("unique_id", observed=True)
        .tail(row_count)
        .reset_index(drop=True)
    )


In [ ]:
# ml/common.py
def log_mlflow_datasets(
    *,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    dataset_prefix: str,
    artifact_prefix: str,
    validation_df: pd.DataFrame | None = None,
) -> None:
    datasets = [
        ("train", "training", train_df),
        ("validation", "validation", validation_df),
        ("test", "evaluation", test_df),
    ]
    for split_name, context, dataset_df in datasets:
        if dataset_df is None or dataset_df.empty:
            continue

        dataset_name = f"{dataset_prefix}_{split_name}".replace("/", "_")
        mlflow.log_input(
            mlflow.data.from_pandas(dataset_df, name=dataset_name),
            context=context,
        )
        mlflow.log_table(
            dataset_df,
            f"{artifact_prefix}/datasets/{split_name}.json",
        )


In [ ]:
# ml/common.py
def _prediction_frame(
    test_df: pd.DataFrame,
    predictions: pd.DataFrame,
) -> pd.DataFrame:
    expected_dates = (
        test_df[["unique_id", "ds"]]
        .sort_values(["unique_id", "ds"])
        .reset_index(drop=True)
    )
    pred_df = predictions.sort_values(["unique_id", "ds"]).reset_index(drop=True)
    pred_df["_forecast_step"] = pred_df.groupby(
        "unique_id",
        observed=True,
    ).cumcount()
    expected_dates["_forecast_step"] = expected_dates.groupby(
        "unique_id",
        observed=True,
    ).cumcount()

    aligned = (
        pred_df.drop(columns=["ds"])
        .merge(
            expected_dates,
            on=["unique_id", "_forecast_step"],
            how="left",
            validate="one_to_one",
        )
        .drop(columns=["_forecast_step"])
    )
    return aligned.merge(
        test_df[["unique_id", "ds", "y"]],
        on=["unique_id", "ds"],
        how="left",
        validate="one_to_one",
    )


In [ ]:
# ml/common.py
def _regression_metrics(joined_df: pd.DataFrame) -> pd.DataFrame:
    models = [
        column
        for column in joined_df.columns
        if column
        not in {
            "unique_id",
            "ds",
            "y",
            "previous_actual_close",
            "actual_long",
        }
        and "-lo-" not in column
        and "-hi-" not in column
    ]
    rows = []
    for model in models:
        valid_rows = joined_df[["y", model]].dropna()
        if valid_rows.empty:
            continue

        mse = mean_squared_error(valid_rows["y"], valid_rows[model])
        non_zero_actuals = valid_rows[valid_rows["y"] != 0]
        mape = (
            float(
                np.mean(
                    np.abs(
                        (non_zero_actuals["y"] - non_zero_actuals[model])
                        / non_zero_actuals["y"]
                    )
                )
                * 100.0
            )
            if not non_zero_actuals.empty
            else None
        )
        rows.append(
            {
                "model": model,
                "mae": mean_absolute_error(valid_rows["y"], valid_rows[model]),
                "rmse": mse**0.5,
                "mape": mape,
                "r2": (
                    r2_score(valid_rows["y"], valid_rows[model])
                    if len(valid_rows) >= 2
                    else None
                ),
                "rows": len(valid_rows),
            }
        )
    return pd.DataFrame(rows)


## MLForecast Models


In [ ]:
# ml/mlforecast_models.py
def _worker_count() -> int:
    return cpu_count_from_env("MODEL_N_JOBS")


In [ ]:
# ml/mlforecast_models.py
def _max_estimators() -> int:
    return max(50, int(os.getenv("MODEL_MAX_ESTIMATORS", "300")))


In [ ]:
# ml/mlforecast_models.py
def _estimator_verbose() -> bool:
    return bool_from_env("MODEL_ESTIMATOR_VERBOSE", default=False)


In [ ]:
# ml/mlforecast_models.py
def ridge_config(trial: optuna.Trial) -> dict:
    return {
        "alpha": trial.suggest_float("alpha", 0.01, 10.0, log=True),
    }


In [ ]:
# ml/mlforecast_models.py
def random_forest_config(trial: optuna.Trial) -> dict:
    max_estimators = _max_estimators()
    return {
        "n_estimators": trial.suggest_int("n_estimators", 50, max_estimators),
        "max_depth": trial.suggest_int("max_depth", 2, 24),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 25),
        "max_features": trial.suggest_float("max_features", 0.5, 1.0),
        "random_state": 26,
        "n_jobs": _worker_count(),
        "verbose": 1 if _estimator_verbose() else 0,
    }


In [ ]:
# ml/mlforecast_models.py
def lightgbm_config(trial: optuna.Trial) -> dict:
    max_estimators = _max_estimators()
    max_depth = trial.suggest_int("max_depth", 2, 10)
    max_leaves = min(64, (2**max_depth) - 1)
    return {
        "n_estimators": trial.suggest_int("n_estimators", 50, max_estimators),
        "max_depth": max_depth,
        "num_leaves": trial.suggest_int("num_leaves", 2, max_leaves),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 2, 50),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "objective": "regression",
        "force_col_wise": True,
        "random_state": 26,
        "n_jobs": _worker_count(),
        "verbosity": -1,
    }


In [ ]:
# ml/mlforecast_models.py
def xgboost_config(trial: optuna.Trial) -> dict:
    max_estimators = _max_estimators()
    return {
        "n_estimators": trial.suggest_int("n_estimators", 50, max_estimators),
        "max_depth": trial.suggest_int("max_depth", 2, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "objective": "reg:squarederror",
        "random_state": 26,
        "n_jobs": _worker_count(),
        "verbosity": 1 if _estimator_verbose() else 0,
    }


In [ ]:
# ml/mlforecast_models.py
def catboost_config(trial: optuna.Trial) -> dict:
    max_estimators = _max_estimators()
    return {
        "iterations": trial.suggest_int("iterations", 50, max_estimators),
        "depth": trial.suggest_int("depth", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-8, 10.0, log=True),
        "loss_function": "RMSE",
        "random_seed": 26,
        "thread_count": _worker_count(),
        "verbose": _estimator_verbose(),
        "allow_writing_files": False,
    }


In [ ]:
# ml/mlforecast_models.py
def init_config(trial: optuna.Trial) -> dict:
    lags_name = trial.suggest_categorical(
        "lags",
        ["1", "1_7", "1_7_20", "1_7_20_60"],
    )
    lags_map = {
        "1": [1],
        "1_7": [1, 7],
        "1_7_20": [1, 7, 20],
        "1_7_20_60": [1, 7, 20, 60],
    }
    return {"lags": lags_map[lags_name]}


In [ ]:
# ml/mlforecast_models.py
def fit_config(trial: optuna.Trial) -> dict:
    return {"static_features": []}


In [ ]:
# ml/mlforecast_models.py
def build_auto_models() -> dict:
    return {
        "Ridge": AutoRidge(config=ridge_config),
        "RandomForest": AutoRandomForest(config=random_forest_config),
        "LightGBM": AutoLightGBM(config=lightgbm_config),
        "XGBoost": AutoXGBoost(config=xgboost_config),
        "CatBoost": AutoCatboost(config=catboost_config),
    }


In [ ]:
# ml/mlforecast_models.py
def build_custom_auto_models() -> dict:
    return {
        "Ridge": AutoModel(model=Ridge(), config=ridge_config),
        "RandomForest": AutoModel(
            model=RandomForestRegressor(),
            config=random_forest_config,
        ),
        "LightGBM": AutoLightGBM(config=lightgbm_config),
        "XGBoost": AutoXGBoost(config=xgboost_config),
        "CatBoost": AutoCatboost(config=catboost_config),
    }


## MLForecast Training


In [ ]:
# ml/mlforecast_training.py
def to_mlforecast_dataset(df: pl.DataFrame) -> pl.DataFrame:
    exogenous_columns = [
        column for column in df.columns if column not in non_feature_columns()
    ]

    if {"unique_id", "ds", "y"}.issubset(set(df.columns)):
        return (
            df.select(
                pl.col("unique_id").cast(pl.Utf8),
                pl.col("ds").cast(pl.Datetime("us"), strict=False),
                pl.col("y").cast(pl.Float64, strict=False),
                *[
                    pl.col(column).cast(pl.Float64, strict=False)
                    for column in exogenous_columns
                ],
            )
            .sort(["unique_id", "ds"])
        )

    if not {"symbol", "date", "close"}.issubset(set(df.columns)):
        raise ValueError(
            "MLForecast training data needs either unique_id/ds/y "
            "or symbol/date/close columns."
        )

    return (
        df.select(
            pl.col("symbol").cast(pl.Utf8).alias("unique_id"),
            pl.col("date").cast(pl.Datetime("us"), strict=False).alias("ds"),
            pl.col("close").cast(pl.Float64, strict=False).alias("y"),
            *[
                pl.col(column).cast(pl.Float64, strict=False)
                for column in exogenous_columns
            ],
        )
        .sort(["unique_id", "ds"])
    )


In [ ]:
# ml/mlforecast_training.py
def _business_day_grid(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.group_by("unique_id")
        .agg(
            pl.datetime_ranges(
                pl.col("ds").min(),
                pl.col("ds").max(),
                interval="1d",
            ).alias("ds")
        )
        .explode("ds")
        .filter(pl.col("ds").dt.weekday() <= 5)
    )


In [ ]:
# ml/mlforecast_training.py
def fill_mlforecast_business_day_gaps(df: pl.DataFrame) -> pl.DataFrame:
    if df.is_empty():
        return df

    feature_columns = [
        column for column in df.columns if column not in model_id_columns()
    ]
    source = (
        df.with_row_index("_source_order")
        .sort(["unique_id", "ds", "_source_order"])
        .unique(subset=["unique_id", "ds"], keep="last", maintain_order=True)
        .drop("_source_order")
    )
    filled = (
        _business_day_grid(source)
        .join(source, on=["unique_id", "ds"], how="left")
        .with_columns(pl.col("y").is_null().alias("_synthetic_row"))
        .sort(["unique_id", "ds"])
    )
    if "calendar_gap_days" in feature_columns:
        filled = (
            filled.with_columns(
                (~pl.col("_synthetic_row")).alias("_actual_row")
            )
            .with_columns(
                pl.col("_actual_row")
                .cast(pl.Int64)
                .cum_sum()
                .over("unique_id")
                .alias("_actual_segment")
            )
            .with_columns(
                (
                    pl.col("ds")
                    .cum_count()
                    .over(["unique_id", "_actual_segment"])
                    - 1
                )
                .cast(pl.Int64)
                .alias("_business_gap_run")
            )
        )

    fill_columns = ["y", *feature_columns]
    filled = filled.with_columns(
        [
            pl.col(column)
            .forward_fill()
            .backward_fill()
            .over("unique_id")
            .alias(column)
            for column in fill_columns
            if column != "calendar_gap_days"
        ]
    )
    if "calendar_gap_days" in feature_columns:
        filled = filled.with_columns(
            pl.when(pl.col("_synthetic_row"))
            .then(pl.col("_business_gap_run"))
            .when(pl.col("_business_gap_run").shift(1).over("unique_id") > 0)
            .then(pl.col("_business_gap_run").shift(1).over("unique_id"))
            .otherwise(pl.col("calendar_gap_days"))
            .fill_null(0)
            .cast(pl.Float64)
            .alias("calendar_gap_days")
        )

    return (
        filled.drop(
            [
                column
                for column in [
                    "_synthetic_row",
                    "_source_row",
                    "_actual_row",
                    "_actual_segment",
                    "_business_gap_run",
                ]
                if column in filled.columns
            ]
        )
        .drop_nulls(["unique_id", "ds", "y", *feature_columns])
        .filter(pl.all_horizontal(pl.col(["y", *feature_columns]).is_finite()))
        .select(["unique_id", "ds", "y", *feature_columns])
        .sort(["unique_id", "ds"])
    )


In [ ]:
# ml/mlforecast_training.py
def to_mlforecast_frame(df: pl.DataFrame) -> pd.DataFrame:
    model_df = fill_mlforecast_business_day_gaps(to_mlforecast_dataset(df))
    return model_df.to_pandas()


In [ ]:
# ml/mlforecast_training.py
def make_train_test_split(
    dataset: pl.DataFrame,
    test_horizon: int,
) -> dict[str, pd.DataFrame]:
    model_df = to_mlforecast_frame(dataset)
    train_df, test_df = split_train_test_by_horizon(model_df, test_horizon)
    return {
        "full": model_df,
        "train": train_df,
        "test": test_df,
    }


In [ ]:
# ml/mlforecast_training.py
def build_auto_mlforecast(freq: str = "B") -> Any:
    try:
        from mlforecast.auto import AutoMLForecast
    except OSError as exc:
        if "libgomp.so.1" not in str(exc):
            raise
        raise RuntimeError(
            "MLForecast cannot start because LightGBM needs libgomp.so.1. "
            "Rebuild the devcontainer, or run inside the container: "
            "`apt-get update && apt-get install -y libgomp1`."
        ) from exc

    # NOTE: project package import omitted; source appears in notebook cells.

    return AutoMLForecast(
        models=build_auto_models(),
        freq=freq,
        init_config=init_config,
        fit_config=fit_config,
        num_threads=cpu_count_from_env("MLFORECAST_NUM_THREADS"),
        reuse_cv_splits=True,
    )


In [ ]:
# ml/mlforecast_training.py
def build_auto_mlforecast_spec(
    *,
    freq: str = "B",
    validation_horizon: int = 1,
    test_horizon: int = 5,
    n_windows: int = 3,
    n_trials: int = 20,
    level: list[int] | None = None,
    verbose: bool = True,
    models: list[str] | None = None,
    tier_name: str = "tier1",
) -> dict[str, Any]:
    return {
        "tier_name": tier_name,
        "freq": freq,
        "validation_horizon": validation_horizon,
        "test_horizon": test_horizon,
        "n_windows": n_windows,
        "n_trials": n_trials,
        "level": level or [80, 95],
        "verbose": verbose,
        "models": models,
    }


In [ ]:
# ml/mlforecast_training.py
def _trial_logger(study: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
    LOGGER.info(
        "Optuna trial finished | study=%s trial=%s state=%s value=%s params=%s",
        study.study_name,
        trial.number,
        trial.state.name,
        trial.value,
        trial.params,
    )


In [ ]:
# ml/mlforecast_training.py
def train_auto_mlforecast_models_from_split(
    train_test_split: dict[str, pd.DataFrame],
    *,
    model_spec: dict[str, Any] | None = None,
    freq: str = "B",
    validation_horizon: int = 1,
    test_horizon: int = 5,
    n_windows: int = 3,
    n_trials: int = 20,
    level: list[int] | None = None,
    verbose: bool = True,
    study_kwargs: dict[str, Any] | None = None,
    optimize_kwargs: dict[str, Any] | None = None,
) -> dict[str, Any]:
    filter_sklearn_parallel_warnings()
    tier_name = "tier1"
    if model_spec:
        tier_name = model_spec.get("tier_name", tier_name)
        freq = model_spec.get("freq", freq)
        validation_horizon = model_spec.get(
            "validation_horizon",
            validation_horizon,
        )
        test_horizon = model_spec.get("test_horizon", test_horizon)
        n_windows = model_spec.get("n_windows", n_windows)
        n_trials = model_spec.get("n_trials", n_trials)
        level = model_spec.get("level", level)
        verbose = model_spec.get("verbose", verbose)

    level = level or [80, 95]
    os.environ["MLFORECAST_VERBOSE"] = "1" if verbose else "0"
    optuna.logging.set_verbosity(
        optuna.logging.INFO if verbose else optuna.logging.WARNING
    )
    study_kwargs = study_kwargs or {
        "study_name": "stock_close_automlforecast",
    }
    study_kwargs.pop("direction", None)
    optimize_kwargs = optimize_kwargs or {}
    callbacks = list(optimize_kwargs.get("callbacks", []))
    if verbose:
        callbacks.append(_trial_logger)
    optimize_kwargs["callbacks"] = callbacks

    train_df = train_test_split["train"]
    test_df = train_test_split["test"]
    auto_mlf = build_auto_mlforecast(freq=freq)
    model_names = (
        model_spec.get("models")
        if model_spec and model_spec.get("models")
        else list(auto_mlf.models.keys())
    )

    LOGGER.info(
        "Starting AutoMLForecast training | rows=%s test_rows=%s models=%s "
        "freq=%s n_windows=%s validation_horizon=%s n_trials=%s",
        len(train_df),
        len(test_df),
        model_names,
        freq,
        n_windows,
        validation_horizon,
        n_trials,
    )

    configure_mlflow_tracking(experiment_name=tier_experiment_name(tier_name))

    with mlflow.start_run(
        run_name=f"stock-close-{tier_name}-automlforecast",
        nested=True,
    ):
        dynamic_feature_columns = [
            column
            for column in train_df.columns
            if column not in {"unique_id", "ds", "y"}
        ]

        mlflow.log_params(
            {
                "freq": freq,
                "validation_horizon": validation_horizon,
                "test_horizon": test_horizon,
                "n_windows": n_windows,
                "n_trials": n_trials,
                "verbose": verbose,
                "tier_name": tier_name,
                "model_input_columns": "unique_id,ds,y",
                "dynamic_features": ",".join(dynamic_feature_columns),
            }
        )
        log_mlflow_datasets(
            train_df=train_df,
            validation_df=validation_reference_frame(
                train_df,
                validation_horizon=validation_horizon,
                n_windows=n_windows,
            ),
            test_df=test_df,
            dataset_prefix=f"stock_close_{tier_name}_mlforecast",
            artifact_prefix=f"mlforecast/{tier_name}",
        )

        from mlforecast.utils import PredictionIntervals

        auto_mlf.fit(
            df=train_df,
            n_windows=n_windows,
            h=validation_horizon,
            num_samples=n_trials,
            id_col="unique_id",
            time_col="ds",
            target_col="y",
            study_kwargs=study_kwargs,
            optimize_kwargs=optimize_kwargs,
            prediction_intervals=PredictionIntervals(
                n_windows=n_windows,
                h=validation_horizon,
            ),
        )

        LOGGER.info("AutoMLForecast fit completed. Generating test predictions.")

        predict_kwargs = {"h": test_horizon, "level": level}
        if dynamic_feature_columns:
            predict_kwargs["X_df"] = test_df.drop(columns=["y"])

        predictions = auto_mlf.predict(**predict_kwargs)
        joined_df = _prediction_frame(test_df, predictions)
        regression_df = _regression_metrics(joined_df)
        long_direction_df = long_only_directional_metrics(joined_df, train_df)

        LOGGER.info("Regression metrics:\n%s", regression_df.to_string(index=False))
        LOGGER.info(
            "Long-only directional metrics:\n%s",
            long_direction_df.to_string(index=False),
        )

        mlflow.log_table(
            joined_df,
            f"mlforecast/{tier_name}/evaluation/predictions.json",
        )
        mlflow.log_table(
            regression_df,
            f"mlforecast/{tier_name}/evaluation/regression_metrics.json",
        )
        mlflow.log_table(
            long_direction_df,
            f"mlforecast/{tier_name}/evaluation/long_only_direction_metrics.json",
        )
        log_forecast_plots(
            train_df=train_df,
            joined_df=joined_df,
            levels=level,
            artifact_prefix=f"mlforecast/{tier_name}/plots",
        )

        for _, row in regression_df.iterrows():
            for metric_name in ["mae", "rmse", "mape", "r2"]:
                if pd.isna(row.get(metric_name)):
                    continue
                mlflow.log_metric(
                    f"{row['model']}.test.{metric_name}",
                    float(row[metric_name]),
                )

        for _, row in long_direction_df.iterrows():
            if row[["long_accuracy", "long_precision", "long_recall"]].isna().any():
                continue

            mlflow.log_metric(
                f"{row['model']}.long.accuracy",
                float(row["long_accuracy"]),
            )
            mlflow.log_metric(
                f"{row['model']}.long.precision",
                float(row["long_precision"]),
            )
            mlflow.log_metric(
                f"{row['model']}.long.recall",
                float(row["long_recall"]),
            )
            mlflow.log_metric(
                f"{row['model']}.long.signal_rate",
                float(row["long_signal_rate"]),
            )

        import mlforecast.flavor

        for model_name, fitted_model in auto_mlf.models_.items():
            mlforecast.flavor.log_model(
                fitted_model,
                name=f"stock_close_{tier_name}_{model_name}",
                artifact_path=None,
            )

    return {
        "model": auto_mlf,
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "predictions": joined_df,
        "regression_metrics": regression_df,
        "long_direction_metrics": long_direction_df,
    }


In [ ]:
# ml/mlforecast_training.py
def train_auto_mlforecast_models(
    dataset: pl.DataFrame,
    *,
    freq: str = "B",
    validation_horizon: int = 1,
    test_horizon: int = 5,
    n_windows: int = 3,
    n_trials: int = 20,
    level: list[int] | None = None,
    study_kwargs: dict[str, Any] | None = None,
    optimize_kwargs: dict[str, Any] | None = None,
) -> dict[str, Any]:
    train_test_split = make_train_test_split(
        dataset,
        test_horizon=test_horizon,
    )
    return train_auto_mlforecast_models_from_split(
        train_test_split,
        freq=freq,
        validation_horizon=validation_horizon,
        test_horizon=test_horizon,
        n_windows=n_windows,
        n_trials=n_trials,
        level=level,
        study_kwargs=study_kwargs,
        optimize_kwargs=optimize_kwargs,
    )


## StatsForecast Models


In [ ]:
# ml/stats_models.py
def _model_class(class_name: str):
    if class_name not in SUPPORTED_STATSFORECAST_MODELS:
        raise ValueError(
            f"Unsupported StatsForecast model {class_name!r}. "
            f"Supported models: {sorted(SUPPORTED_STATSFORECAST_MODELS)}"
        )
    try:
        return getattr(stats_models, class_name)
    except AttributeError as exc:
        raise ValueError(
            f"StatsForecast does not expose {class_name!r} in this installation."
        ) from exc


In [ ]:
# ml/stats_models.py
def build_statsforecast_models(
    *,
    seasonal_length: int,
    horizon: int,
    conformal_n_windows: int,
    models_config: list[dict[str, Any]],
) -> list[Any]:
    def prediction_intervals() -> ConformalIntervals:
        return ConformalIntervals(
            h=horizon,
            n_windows=conformal_n_windows,
        )

    def defaults_for(class_name: str) -> dict[str, Any]:
        if class_name in {
            "AutoARIMA",
            "AutoCES",
            "AutoETS",
            "AutoMFLES",
            "AutoTBATS",
            "AutoTheta",
        }:
            return {
                "season_length": seasonal_length,
                "prediction_intervals": prediction_intervals(),
            }
        if class_name == "AutoRegressive":
            return {
                "lags": sorted({1, seasonal_length}),
                "prediction_intervals": prediction_intervals(),
            }
        return {}

    built_models = []
    for model_config in models_config:
        class_name = model_config["class_name"]
        alias = model_config.get("alias", class_name)
        kwargs = model_config.get("kwargs") or {}
        model_class = _model_class(class_name)
        built_models.append(
            model_class(
                **defaults_for(class_name),
                **kwargs,
                alias=alias,
            )
        )

    return built_models


## StatsForecast Training


In [ ]:
# ml/stats_training.py
def build_statsforecast_spec(
    *,
    freq: str = "B",
    seasonal_length: int = 5,
    validation_horizon: int = 1,
    test_horizon: int = 5,
    conformal_n_windows: int = 3,
    level: list[int] | None = None,
    models: list[dict[str, Any]] | None = None,
    verbose: bool = True,
    tier_name: str = "tier1",
) -> dict[str, Any]:
    return {
        "tier_name": tier_name,
        "freq": freq,
        "seasonal_length": seasonal_length,
        "validation_horizon": validation_horizon,
        "test_horizon": test_horizon,
        "conformal_n_windows": conformal_n_windows,
        "level": level or [80, 95],
        "models": models
        or [
            {"class_name": "AutoARIMA", "alias": "AutoARIMA"},
            {"class_name": "AutoETS", "alias": "AutoETS"},
            {"class_name": "AutoTheta", "alias": "AutoTheta"},
            {"class_name": "AutoRegressive", "alias": "AutoRegressive"},
        ],
        "verbose": verbose,
    }


In [ ]:
# ml/stats_training.py
def _model_names_from_predictions(joined_df: pd.DataFrame) -> list[str]:
    return [
        column
        for column in joined_df.columns
        if column not in {"unique_id", "ds", "y"}
        and "-lo-" not in column
        and "-hi-" not in column
    ]


In [ ]:
# ml/stats_training.py
def _pickle_statsforecast_model(model: StatsForecast) -> str:
    temp_dir = tempfile.mkdtemp(prefix="statsforecast_")
    model_path = Path(temp_dir) / "statsforecast_model.pkl"
    with model_path.open("wb") as file_obj:
        pickle.dump(model, file_obj)
    return str(model_path)


In [ ]:
# ml/stats_training.py
def train_statsforecast_models_from_split(
    train_test_split: dict[str, pd.DataFrame],
    *,
    model_spec: dict[str, Any],
) -> dict[str, Any]:
    freq = model_spec["freq"]
    seasonal_length = model_spec["seasonal_length"]
    validation_horizon = model_spec["validation_horizon"]
    test_horizon = model_spec["test_horizon"]
    conformal_n_windows = model_spec["conformal_n_windows"]
    level = model_spec.get("level") or [80, 95]
    verbose = model_spec.get("verbose", True)
    tier_name = model_spec.get("tier_name", "tier1")

    train_df = train_test_split["train"]
    test_df = train_test_split["test"]
    dynamic_feature_columns = [
        column for column in train_df.columns if column not in {"unique_id", "ds", "y"}
    ]
    models = build_statsforecast_models(
        seasonal_length=seasonal_length,
        horizon=validation_horizon,
        conformal_n_windows=conformal_n_windows,
        models_config=model_spec["models"],
    )

    LOGGER.info(
        "Starting StatsForecast training | rows=%s test_rows=%s models=%s "
        "freq=%s seasonal_length=%s conformal_n_windows=%s",
        len(train_df),
        len(test_df),
        [getattr(model, "alias", model.__class__.__name__) for model in models],
        freq,
        seasonal_length,
        conformal_n_windows,
    )

    configure_mlflow_tracking(experiment_name=tier_experiment_name(tier_name))
    with mlflow.start_run(run_name=f"stock-close-{tier_name}-statsforecast", nested=True):
        mlflow.log_params(
            {
                "tier_name": tier_name,
                "freq": freq,
                "seasonal_length": seasonal_length,
                "validation_horizon": validation_horizon,
                "test_horizon": test_horizon,
                "conformal_n_windows": conformal_n_windows,
                "verbose": verbose,
                "models": ",".join(
                    model_config.get("alias", model_config["class_name"])
                    for model_config in model_spec["models"]
                ),
                "dynamic_features": ",".join(dynamic_feature_columns),
            }
        )
        log_mlflow_datasets(
            train_df=train_df,
            validation_df=validation_reference_frame(
                train_df,
                validation_horizon=validation_horizon,
                n_windows=conformal_n_windows,
            ),
            test_df=test_df,
            dataset_prefix=f"stock_close_{tier_name}_statsforecast",
            artifact_prefix=f"statsforecast/{tier_name}",
        )

        sf = StatsForecast(
            models=models,
            freq=freq,
            n_jobs=cpu_count_from_env("MODEL_N_JOBS"),
            verbose=verbose,
        )
        fitted_sf = sf.fit(df=train_df)

        predict_kwargs: dict[str, Any] = {"h": test_horizon, "level": level}
        if dynamic_feature_columns:
            predict_kwargs["X_df"] = test_df.drop(columns=["y"])

        try:
            predictions = fitted_sf.predict(**predict_kwargs)
        except Exception:
            if "X_df" not in predict_kwargs:
                raise
            LOGGER.warning(
                "StatsForecast prediction with X_df failed; retrying without "
                "dynamic features.",
                exc_info=True,
            )
            predictions = fitted_sf.predict(h=test_horizon, level=level)

        if hasattr(predictions, "to_pandas"):
            predictions = predictions.to_pandas()
        joined_df = _prediction_frame(test_df, predictions)
        regression_df = _regression_metrics(joined_df)
        long_direction_df = long_only_directional_metrics(joined_df, train_df)

        LOGGER.info("StatsForecast regression metrics:\n%s", regression_df)
        LOGGER.info("StatsForecast long metrics:\n%s", long_direction_df)

        mlflow.log_table(
            joined_df,
            f"statsforecast/{tier_name}/evaluation/predictions.json",
        )
        mlflow.log_table(
            regression_df,
            f"statsforecast/{tier_name}/evaluation/regression_metrics.json",
        )
        mlflow.log_table(
            long_direction_df,
            f"statsforecast/{tier_name}/evaluation/long_only_direction_metrics.json",
        )
        log_forecast_plots(
            train_df=train_df,
            joined_df=joined_df,
            levels=level,
            artifact_prefix=f"statsforecast/{tier_name}/plots",
        )

        for _, row in regression_df.iterrows():
            for metric_name in ["mae", "rmse", "mape", "r2"]:
                if pd.isna(row.get(metric_name)):
                    continue
                mlflow.log_metric(
                    f"statsforecast.{row['model']}.test.{metric_name}",
                    float(row[metric_name]),
                )

        for _, row in long_direction_df.iterrows():
            if row[["long_accuracy", "long_precision", "long_recall"]].isna().any():
                continue
            mlflow.log_metric(
                f"statsforecast.{row['model']}.long.accuracy",
                float(row["long_accuracy"]),
            )
            mlflow.log_metric(
                f"statsforecast.{row['model']}.long.precision",
                float(row["long_precision"]),
            )
            mlflow.log_metric(
                f"statsforecast.{row['model']}.long.recall",
                float(row["long_recall"]),
            )
            mlflow.log_metric(
                f"statsforecast.{row['model']}.long.signal_rate",
                float(row["long_signal_rate"]),
            )

        model_path = _pickle_statsforecast_model(fitted_sf)
        mlflow.log_artifact(
            model_path,
            artifact_path=f"statsforecast/{tier_name}/model",
        )

    return {
        "model": fitted_sf,
        "model_names": _model_names_from_predictions(joined_df),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "predictions": joined_df,
        "regression_metrics": regression_df,
        "long_direction_metrics": long_direction_df,
    }


## Feast And Timescale Serving


In [ ]:
# serving/feast_store.py
def _ensure_feature_repo_on_path() -> None:
    (FEATURE_REPO_DIR / "data").mkdir(parents=True, exist_ok=True)
    feature_repo_parent = str(FEATURE_REPO_DIR.parent)
    if feature_repo_parent not in sys.path:
        sys.path.insert(0, feature_repo_parent)


In [ ]:
# serving/feast_store.py
def _timescale_connection_kwargs() -> dict[str, str | int]:
    return {
        "host": os.getenv("TIMESCALE_HOST", "host.docker.internal"),
        "port": int(os.getenv("TIMESCALE_PORT", "5432")),
        "dbname": os.getenv("TIMESCALE_DB", "dataops"),
        "user": os.getenv("TIMESCALE_USER", "dataops"),
        "password": os.getenv("TIMESCALE_PASSWORD", "dataops"),
    }


In [ ]:
# serving/feast_store.py
def _schema_name(table_name: str) -> str:
    return table_name.split(".", maxsplit=1)[0]


In [ ]:
# serving/feast_store.py
def _create_timescale_feature_table(cursor) -> None:
    cursor.execute(f"CREATE SCHEMA IF NOT EXISTS {_schema_name(TIMESCALE_TABLE)};")
    cursor.execute(
        f"""
        CREATE TABLE IF NOT EXISTS {TIMESCALE_TABLE} (
            symbol TEXT NOT NULL,
            "date" TIMESTAMPTZ NOT NULL,
            created_timestamp TIMESTAMPTZ NOT NULL,
            prev_open DOUBLE PRECISION,
            prev_high DOUBLE PRECISION,
            prev_low DOUBLE PRECISION,
            prev_volume DOUBLE PRECISION,
            calendar_gap_days INTEGER,
            month_sin_1 DOUBLE PRECISION,
            month_cos_1 DOUBLE PRECISION,
            month_sin_2 DOUBLE PRECISION,
            month_cos_2 DOUBLE PRECISION,
            day_sin_1 DOUBLE PRECISION,
            day_cos_1 DOUBLE PRECISION,
            day_sin_2 DOUBLE PRECISION,
            day_cos_2 DOUBLE PRECISION,
            day_of_year_sin_1 DOUBLE PRECISION,
            day_of_year_cos_1 DOUBLE PRECISION,
            day_of_year_sin_2 DOUBLE PRECISION,
            day_of_year_cos_2 DOUBLE PRECISION,
            PRIMARY KEY (symbol, "date")
        );
        """
    )
    cursor.execute(
        f"""
        ALTER TABLE {TIMESCALE_TABLE}
            ADD COLUMN IF NOT EXISTS prev_open DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS prev_high DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS prev_low DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS prev_volume DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS calendar_gap_days INTEGER,
            ADD COLUMN IF NOT EXISTS month_sin_1 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS month_cos_1 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS month_sin_2 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS month_cos_2 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS day_sin_1 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS day_cos_1 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS day_sin_2 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS day_cos_2 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS day_of_year_sin_1 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS day_of_year_cos_1 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS day_of_year_sin_2 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS day_of_year_cos_2 DOUBLE PRECISION;
        """
    )
    cursor.execute("CREATE EXTENSION IF NOT EXISTS timescaledb;")
    cursor.execute(
        """
        SELECT create_hypertable(
            %s,
            'date',
            if_not_exists => TRUE
        );
        """,
        (TIMESCALE_TABLE,),
    )


In [ ]:
# serving/feast_store.py
def _create_timescale_close_model_dataset_table(cursor) -> None:
    cursor.execute(
        f"CREATE SCHEMA IF NOT EXISTS {_schema_name(TIMESCALE_CLOSE_MODEL_DATASET_TABLE)};"
    )
    cursor.execute(
        f"""
        CREATE TABLE IF NOT EXISTS {TIMESCALE_CLOSE_MODEL_DATASET_TABLE} (
            unique_id TEXT NOT NULL,
            ds TIMESTAMPTZ NOT NULL,
            y DOUBLE PRECISION NOT NULL,
            month_sin_1 DOUBLE PRECISION,
            month_cos_1 DOUBLE PRECISION,
            day_sin_1 DOUBLE PRECISION,
            day_cos_1 DOUBLE PRECISION,
            day_of_year_sin_1 DOUBLE PRECISION,
            day_of_year_cos_1 DOUBLE PRECISION,
            created_timestamp TIMESTAMPTZ NOT NULL,
            PRIMARY KEY (unique_id, ds)
        );
        """
    )
    cursor.execute(
        f"""
        ALTER TABLE {TIMESCALE_CLOSE_MODEL_DATASET_TABLE}
            ADD COLUMN IF NOT EXISTS month_sin_1 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS month_cos_1 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS day_sin_1 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS day_cos_1 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS day_of_year_sin_1 DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS day_of_year_cos_1 DOUBLE PRECISION;
        """
    )
    cursor.execute("CREATE EXTENSION IF NOT EXISTS timescaledb;")
    cursor.execute(
        """
        SELECT create_hypertable(
            %s,
            'ds',
            if_not_exists => TRUE
        );
        """,
        (TIMESCALE_CLOSE_MODEL_DATASET_TABLE,),
    )


In [ ]:
# serving/feast_store.py
def _postgres_type_for_conventional_gap_column(column: str) -> str:
    if column in {"symbol", "Gap_Type"}:
        return "TEXT"
    if column in {"date", "created_timestamp"}:
        return "TIMESTAMPTZ"
    if column in {"month", "day", "day_of_year", "calendar_gap_days"}:
        return "INTEGER"
    if column in CONDITION_COLUMNS:
        return "BOOLEAN"
    return "DOUBLE PRECISION"


In [ ]:
# serving/feast_store.py
def _create_timescale_conventional_gap_trading_table(cursor) -> None:
    cursor.execute(
        f"CREATE SCHEMA IF NOT EXISTS {_schema_name(TIMESCALE_CONVENTIONAL_GAP_TRADING_TABLE)};"
    )
    column_definitions = ",\n            ".join(
        f'"{column}" {_postgres_type_for_conventional_gap_column(column)}'
        for column in CONVENTIONAL_GAP_TRADING_COLUMNS
    )
    alter_columns = ",\n            ".join(
        f'ADD COLUMN IF NOT EXISTS "{column}" {_postgres_type_for_conventional_gap_column(column)}'
        for column in CONVENTIONAL_GAP_TRADING_COLUMNS
        if column not in {"symbol", "date"}
    )
    cursor.execute(
        f"""
        CREATE TABLE IF NOT EXISTS {TIMESCALE_CONVENTIONAL_GAP_TRADING_TABLE} (
            {column_definitions},
            PRIMARY KEY (symbol, "date")
        );
        """
    )
    cursor.execute(
        f"""
        ALTER TABLE {TIMESCALE_CONVENTIONAL_GAP_TRADING_TABLE}
            {alter_columns};
        """
    )
    cursor.execute("CREATE EXTENSION IF NOT EXISTS timescaledb;")
    cursor.execute(
        """
        SELECT create_hypertable(
            %s,
            'date',
            if_not_exists => TRUE
        );
        """,
        (TIMESCALE_CONVENTIONAL_GAP_TRADING_TABLE,),
    )


In [ ]:
# serving/feast_store.py
def _create_timescale_pecnet_preprocessed_table(cursor) -> None:
    cursor.execute(
        f"CREATE SCHEMA IF NOT EXISTS {_schema_name(TIMESCALE_PECNET_PREPROCESSED_TABLE)};"
    )
    cursor.execute(
        f"""
        CREATE TABLE IF NOT EXISTS {TIMESCALE_PECNET_PREPROCESSED_TABLE} (
            row_key TEXT NOT NULL,
            tier TEXT NOT NULL,
            symbol TEXT NOT NULL,
            event_timestamp TIMESTAMPTZ NOT NULL,
            split TEXT NOT NULL,
            split_index INTEGER NOT NULL,
            variable_name TEXT NOT NULL,
            variable_index INTEGER NOT NULL,
            sample_index INTEGER NOT NULL,
            step_index INTEGER NOT NULL,
            value DOUBLE PRECISION,
            target_y DOUBLE PRECISION,
            created_timestamp TIMESTAMPTZ NOT NULL,
            PRIMARY KEY (row_key, event_timestamp)
        );
        """
    )
    cursor.execute(
        f"""
        ALTER TABLE {TIMESCALE_PECNET_PREPROCESSED_TABLE}
            ADD COLUMN IF NOT EXISTS tier TEXT,
            ADD COLUMN IF NOT EXISTS symbol TEXT,
            ADD COLUMN IF NOT EXISTS event_timestamp TIMESTAMPTZ,
            ADD COLUMN IF NOT EXISTS split TEXT,
            ADD COLUMN IF NOT EXISTS split_index INTEGER,
            ADD COLUMN IF NOT EXISTS variable_name TEXT,
            ADD COLUMN IF NOT EXISTS variable_index INTEGER,
            ADD COLUMN IF NOT EXISTS sample_index INTEGER,
            ADD COLUMN IF NOT EXISTS step_index INTEGER,
            ADD COLUMN IF NOT EXISTS value DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS target_y DOUBLE PRECISION,
            ADD COLUMN IF NOT EXISTS created_timestamp TIMESTAMPTZ;
        """
    )
    cursor.execute("CREATE EXTENSION IF NOT EXISTS timescaledb;")
    cursor.execute(
        """
        SELECT create_hypertable(
            %s,
            'event_timestamp',
            if_not_exists => TRUE
        );
        """,
        (TIMESCALE_PECNET_PREPROCESSED_TABLE,),
    )


In [ ]:
# serving/feast_store.py
def _to_pandas_for_feature_store(df: pl.DataFrame) -> pd.DataFrame:
    pdf = df.select(FEAST_OFFLINE_COLUMNS).to_pandas()
    pdf["date"] = pd.to_datetime(pdf["date"], utc=True)
    pdf["created_timestamp"] = pd.to_datetime(pdf["created_timestamp"], utc=True)
    return pdf.where(pd.notnull(pdf), None)


In [ ]:
# serving/feast_store.py
def _model_feature_date_key(date: pd.Series) -> pd.Series:
    return pd.to_datetime(date, utc=True).dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")


In [ ]:
# serving/feast_store.py
def _model_feature_key(symbol: pd.Series, date_key: pd.Series) -> pd.Series:
    return symbol.astype(str) + "|" + date_key.astype(str)


In [ ]:
# serving/feast_store.py
def _to_pandas_for_tier2_feature_dataset(df: pl.DataFrame) -> pd.DataFrame:
    pdf = df.select(FEAST_OFFLINE_COLUMNS).to_pandas()
    pdf["date"] = pd.to_datetime(pdf["date"], utc=True)
    pdf["date_key"] = _model_feature_date_key(pdf["date"])
    pdf["feature_key"] = _model_feature_key(pdf["symbol"], pdf["date_key"])
    pdf["created_timestamp"] = pd.to_datetime(pdf["created_timestamp"], utc=True)
    return pdf.where(pd.notnull(pdf), None)


In [ ]:
# serving/feast_store.py
def _iter_polars_row_batches(
    df: pl.DataFrame,
    columns: list[str],
    batch_size: int | None = None,
):
    if batch_size is None:
        batch_size = TIMESCALE_WRITE_BATCH_SIZE
    selected = df.select(columns)
    for batch in selected.iter_slices(n_rows=batch_size):
        rows = list(batch.iter_rows(named=False))
        if rows:
            yield rows


In [ ]:
# serving/feast_store.py
def _fill_interval() -> str:
    if TIMESCALE_DAILY_FILL_FREQ.upper() == "B":
        return "1d"
    return TIMESCALE_DAILY_FILL_FREQ


In [ ]:
# serving/feast_store.py
def _time_encoding_expressions(
    time_column: str,
    output_columns: list[str],
) -> list[pl.Expr]:
    expressions = []
    time_parts = {
        "month": (pl.col(time_column).dt.month(), 12.0),
        "day": (pl.col(time_column).dt.day(), 31.0),
        "day_of_year": (pl.col(time_column).dt.ordinal_day(), 366.0),
    }

    for column_name, (time_expr, period) in time_parts.items():
        for harmonic in (1, 2):
            angle = 2.0 * pi * harmonic * time_expr.cast(pl.Float64) / period
            sin_column = f"{column_name}_sin_{harmonic}"
            cos_column = f"{column_name}_cos_{harmonic}"
            if sin_column in output_columns:
                expressions.append(angle.sin().alias(sin_column))
            if cos_column in output_columns:
                expressions.append(angle.cos().alias(cos_column))

    return expressions


In [ ]:
# serving/feast_store.py
def _fill_daily_gaps(
    df: pl.DataFrame,
    *,
    id_column: str,
    time_column: str,
    output_columns: list[str],
    preserve_calendar_gap_days: bool = False,
) -> pl.DataFrame:
    if df.is_empty():
        return df.select(output_columns)

    source = (
        df.select(output_columns)
        .with_columns(
            pl.col(time_column)
            .cast(pl.Datetime("us"), strict=False)
            .dt.truncate("1d")
            .alias(time_column)
        )
        .with_row_index("_source_order")
        .sort([id_column, time_column, "_source_order"])
        .unique(subset=[id_column, time_column], keep="last", maintain_order=True)
        .drop("_source_order")
        .with_columns(pl.lit(True).alias("_source_row"))
    )

    date_grid = (
        source.group_by(id_column)
        .agg(
            pl.datetime_ranges(
                pl.col(time_column).min(),
                pl.col(time_column).max(),
                interval=_fill_interval(),
            ).alias(time_column)
        )
        .explode(time_column)
    )

    if TIMESCALE_DAILY_FILL_FREQ.upper() == "B":
        date_grid = date_grid.filter(pl.col(time_column).dt.weekday() <= 5)

    filled = (
        date_grid.join(source, on=[id_column, time_column], how="left")
        .with_columns(pl.col("_source_row").is_null().alias("_synthetic_row"))
        .sort([id_column, time_column])
    )
    if preserve_calendar_gap_days and "calendar_gap_days" in output_columns:
        filled = (
            filled.with_columns(
                (~pl.col("_synthetic_row")).alias("_actual_row")
            )
            .with_columns(
                pl.col("_actual_row")
                .cast(pl.Int64)
                .cum_sum()
                .over(id_column)
                .alias("_actual_segment")
            )
            .with_columns(
                (
                    pl.col(time_column)
                    .cum_count()
                    .over([id_column, "_actual_segment"])
                    - 1
                )
                .cast(pl.Int64)
                .alias("_business_gap_run")
            )
        )

    fill_columns = [
        column
        for column in output_columns
        if column not in {id_column, time_column, "created_timestamp"}
        and not (
            preserve_calendar_gap_days
            and column == "calendar_gap_days"
        )
    ]
    fill_expressions = [
        pl.col(column)
        .forward_fill()
        .backward_fill()
        .over(id_column)
        .alias(column)
        for column in fill_columns
    ]

    if preserve_calendar_gap_days and "calendar_gap_days" in output_columns:
        fill_expressions.append(
            pl.when(pl.col("_synthetic_row"))
            .then(pl.col("_business_gap_run"))
            .when(pl.col("_business_gap_run").shift(1).over(id_column) > 0)
            .then(pl.col("_business_gap_run").shift(1).over(id_column))
            .otherwise(pl.col("calendar_gap_days"))
            .fill_null(0)
            .cast(pl.Int32)
            .alias("calendar_gap_days")
        )

    if "created_timestamp" in output_columns:
        fill_expressions.append(
            pl.lit(datetime.now(timezone.utc)).alias("created_timestamp")
        )

    filled = filled.with_columns(fill_expressions)
    time_expressions = _time_encoding_expressions(time_column, output_columns)
    if time_expressions:
        filled = filled.with_columns(time_expressions)

    return filled.select(output_columns)


In [ ]:
# serving/feast_store.py
def _fill_model_feature_daily_gaps(df: pl.DataFrame) -> pl.DataFrame:
    return _fill_daily_gaps(
        df,
        id_column="symbol",
        time_column="date",
        output_columns=FEAST_OFFLINE_COLUMNS,
        preserve_calendar_gap_days=True,
    )


In [ ]:
# serving/feast_store.py
def _fill_close_model_dataset_daily_gaps(df: pl.DataFrame) -> pl.DataFrame:
    return _fill_daily_gaps(
        df,
        id_column="unique_id",
        time_column="ds",
        output_columns=CLOSE_MODEL_DATASET_COLUMNS,
    )


In [ ]:
# serving/feast_store.py
def _write_model_features_to_timescale(df: pl.DataFrame) -> int:
    if df.is_empty():
        return 0

    df = _fill_model_feature_daily_gaps(df)
    update_columns = [
        column
        for column in FEAST_OFFLINE_COLUMNS
        if column not in {"symbol", "date"}
    ]
    quoted_columns = ", ".join(f'"{column}"' for column in FEAST_OFFLINE_COLUMNS)
    update_assignments = ", ".join(
        f'"{column}" = EXCLUDED."{column}"' for column in update_columns
    )

    insert_sql = f"""
        INSERT INTO {TIMESCALE_TABLE} ({quoted_columns})
        VALUES %s
        ON CONFLICT (symbol, "date")
        DO UPDATE SET {update_assignments};
    """

    with psycopg2.connect(**_timescale_connection_kwargs()) as connection:
        with connection.cursor() as cursor:
            _create_timescale_feature_table(cursor)
            symbols = df.get_column("symbol").unique().to_list()
            cursor.execute(
                f"""
                DELETE FROM {TIMESCALE_TABLE}
                WHERE symbol = ANY(%s);
                """,
                (symbols,),
            )
            total_rows = 0
            for rows in _iter_polars_row_batches(df, FEAST_OFFLINE_COLUMNS):
                execute_values(
                    cursor,
                    insert_sql,
                    rows,
                    page_size=TIMESCALE_WRITE_BATCH_SIZE,
                )
                total_rows += len(rows)

    return total_rows


In [ ]:
# serving/feast_store.py
def _to_pandas_for_close_model_dataset(df: pl.DataFrame) -> pd.DataFrame:
    pdf = df.select(CLOSE_MODEL_DATASET_COLUMNS).to_pandas()
    pdf["ds"] = pd.to_datetime(pdf["ds"], utc=True)
    pdf["ds_key"] = _close_model_ds_key(pdf["ds"])
    pdf["series_key"] = _close_model_series_key(pdf["unique_id"], pdf["ds_key"])
    pdf["created_timestamp"] = pd.to_datetime(pdf["created_timestamp"], utc=True)
    return pdf.where(pd.notnull(pdf), None)


In [ ]:
# serving/feast_store.py
def _close_model_ds_key(ds: pd.Series) -> pd.Series:
    return pd.to_datetime(ds, utc=True).dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")


In [ ]:
# serving/feast_store.py
def _close_model_series_key(unique_id: pd.Series, ds_key: pd.Series) -> pd.Series:
    return unique_id.astype(str) + "|" + ds_key.astype(str)


In [ ]:
# serving/feast_store.py
def _write_close_model_dataset_to_timescale(df: pl.DataFrame) -> int:
    if df.is_empty():
        return 0

    df = _fill_close_model_dataset_daily_gaps(df)
    columns = CLOSE_MODEL_DATASET_COLUMNS
    quoted_columns = ", ".join(f'"{column}"' for column in columns)

    insert_sql = f"""
        INSERT INTO {TIMESCALE_CLOSE_MODEL_DATASET_TABLE} ({quoted_columns})
        VALUES %s
        ON CONFLICT (unique_id, ds)
        DO UPDATE SET
            y = EXCLUDED.y,
            month_sin_1 = EXCLUDED.month_sin_1,
            month_cos_1 = EXCLUDED.month_cos_1,
            day_sin_1 = EXCLUDED.day_sin_1,
            day_cos_1 = EXCLUDED.day_cos_1,
            day_of_year_sin_1 = EXCLUDED.day_of_year_sin_1,
            day_of_year_cos_1 = EXCLUDED.day_of_year_cos_1,
            created_timestamp = EXCLUDED.created_timestamp;
    """

    with psycopg2.connect(**_timescale_connection_kwargs()) as connection:
        with connection.cursor() as cursor:
            _create_timescale_close_model_dataset_table(cursor)
            unique_ids = df.get_column("unique_id").unique().to_list()
            cursor.execute(
                f"""
                DELETE FROM {TIMESCALE_CLOSE_MODEL_DATASET_TABLE}
                WHERE unique_id = ANY(%s);
                """,
                (unique_ids,),
            )
            total_rows = 0
            for rows in _iter_polars_row_batches(df, columns):
                execute_values(
                    cursor,
                    insert_sql,
                    rows,
                    page_size=TIMESCALE_WRITE_BATCH_SIZE,
                )
                total_rows += len(rows)

    return total_rows


In [ ]:
# serving/feast_store.py
def _write_conventional_gap_trading_to_timescale(df: pl.DataFrame) -> int:
    if df.is_empty():
        return 0

    columns = CONVENTIONAL_GAP_TRADING_COLUMNS
    update_columns = [
        column
        for column in columns
        if column not in {"symbol", "date"}
    ]
    quoted_columns = ", ".join(f'"{column}"' for column in columns)
    update_assignments = ", ".join(
        f'"{column}" = EXCLUDED."{column}"' for column in update_columns
    )

    insert_sql = f"""
        INSERT INTO {TIMESCALE_CONVENTIONAL_GAP_TRADING_TABLE} ({quoted_columns})
        VALUES %s
        ON CONFLICT (symbol, "date")
        DO UPDATE SET {update_assignments};
    """

    with psycopg2.connect(**_timescale_connection_kwargs()) as connection:
        with connection.cursor() as cursor:
            _create_timescale_conventional_gap_trading_table(cursor)
            total_rows = 0
            for rows in _iter_polars_row_batches(df, columns):
                execute_values(
                    cursor,
                    insert_sql,
                    rows,
                    page_size=TIMESCALE_WRITE_BATCH_SIZE,
                )
                total_rows += len(rows)

    return total_rows


In [ ]:
# serving/feast_store.py
def _to_pandas_for_pecnet_preprocessed(df: pl.DataFrame) -> pd.DataFrame:
    pdf = df.select(PECNET_PREPROCESSED_FEAST_COLUMNS).to_pandas()
    pdf["event_timestamp"] = pd.to_datetime(pdf["event_timestamp"], utc=True)
    pdf["created_timestamp"] = pd.to_datetime(pdf["created_timestamp"], utc=True)
    return pdf.where(pd.notnull(pdf), None)


In [ ]:
# serving/feast_store.py
def _write_pecnet_preprocessed_to_timescale(df: pl.DataFrame) -> int:
    if df.is_empty():
        return 0

    columns = PECNET_PREPROCESSED_COLUMNS
    quoted_columns = ", ".join(f'"{column}"' for column in columns)
    update_columns = [
        column
        for column in columns
        if column not in {"row_key"}
    ]
    update_assignments = ", ".join(
        f'"{column}" = EXCLUDED."{column}"' for column in update_columns
    )
    insert_sql = f"""
        INSERT INTO {TIMESCALE_PECNET_PREPROCESSED_TABLE} ({quoted_columns})
        VALUES %s
        ON CONFLICT (row_key, event_timestamp)
        DO UPDATE SET {update_assignments};
    """

    with psycopg2.connect(**_timescale_connection_kwargs()) as connection:
        with connection.cursor() as cursor:
            _create_timescale_pecnet_preprocessed_table(cursor)
            tiers = df.get_column("tier").unique().to_list()
            symbols = df.get_column("symbol").unique().to_list()
            cursor.execute(
                f"""
                DELETE FROM {TIMESCALE_PECNET_PREPROCESSED_TABLE}
                WHERE tier = ANY(%s)
                  AND symbol = ANY(%s);
                """,
                (tiers, symbols),
            )
            total_rows = 0
            for rows in _iter_polars_row_batches(df, columns):
                execute_values(
                    cursor,
                    insert_sql,
                    rows,
                    page_size=TIMESCALE_WRITE_BATCH_SIZE,
                )
                total_rows += len(rows)

    return total_rows


In [ ]:
# serving/feast_store.py
def _apply_model_feature_definitions() -> FeatureStore:
    _ensure_feature_repo_on_path()
    store = FeatureStore(repo_path=str(FEATURE_REPO_DIR))

    # NOTE: project package import omitted; source appears in notebook cells.

    store.apply(
        [
            ticker,
            stock_model_features_view,
            stock_model_tier_1_feature_service,
            stock_model_tier_2_feature_service,
            stock_model_tier_3_feature_service,
            stock_feature_row,
            stock_model_tier2_dataset_view,
            stock_model_tier2_dataset_service,
        ]
    )
    return store


In [ ]:
# serving/feast_store.py
def _apply_pecnet_preprocessed_definition_and_push(df: pl.DataFrame) -> int:
    if df.is_empty():
        return 0

    _ensure_feature_repo_on_path()
    store = FeatureStore(repo_path=str(FEATURE_REPO_DIR))

    # NOTE: project package import omitted; source appears in notebook cells.

    store.apply(
        [
            pecnet_preprocessed_row,
            pecnet_preprocessed_training_view,
            pecnet_preprocessed_training_service,
        ]
    )
    store.write_to_online_store(
        "pecnet_preprocessed_training_data",
        _to_pandas_for_pecnet_preprocessed(df),
    )
    return len(df)


In [ ]:
# serving/feast_store.py
def _apply_feast_definitions_and_push(df: pl.DataFrame) -> int:
    if df.is_empty():
        return 0

    store = _apply_model_feature_definitions()
    store.write_to_online_store(
        "stock_model_features",
        _to_pandas_for_feature_store(df),
    )
    store.write_to_online_store(
        "stock_model_tier2_dataset",
        _to_pandas_for_tier2_feature_dataset(df),
    )
    return len(df)


In [ ]:
# serving/feast_store.py
def _apply_close_model_dataset_definition() -> None:
    _ensure_feature_repo_on_path()
    store = FeatureStore(repo_path=str(FEATURE_REPO_DIR))

    # NOTE: project package import omitted; source appears in notebook cells.

    store.apply(
        [
            stock_series,
            stock_close_model_dataset_view,
            stock_close_model_dataset_service,
        ]
    )


In [ ]:
# serving/feast_store.py
def _apply_close_model_dataset_definition_and_push(df: pl.DataFrame) -> int:
    if df.is_empty():
        return 0

    df = _fill_close_model_dataset_daily_gaps(df)
    _ensure_feature_repo_on_path()
    store = FeatureStore(repo_path=str(FEATURE_REPO_DIR))

    # NOTE: project package import omitted; source appears in notebook cells.

    store.apply(
        [
            stock_series,
            stock_close_model_dataset_view,
            stock_close_model_dataset_service,
        ]
    )
    store.write_to_online_store(
        "stock_close_model_dataset",
        _to_pandas_for_close_model_dataset(df),
    )
    return len(df)


In [ ]:
# serving/feast_store.py
def publish_close_model_dataset_to_store(df: pl.DataFrame) -> dict[str, object]:
    timescale_rows = _write_close_model_dataset_to_timescale(df)
    feast_online_rows = _apply_close_model_dataset_definition_and_push(df)

    return {
        "timescale_table": TIMESCALE_CLOSE_MODEL_DATASET_TABLE,
        "timescale_rows": timescale_rows,
        "feast_online_rows": feast_online_rows,
        "feast_registry_applied": True,
        "feast_feature_view": "stock_close_model_dataset",
        "model_columns": ["unique_id", "ds", "y", *CLOSE_MODEL_TIME_FEATURE_COLUMNS],
    }


In [ ]:
# serving/feast_store.py
def _read_close_model_entity_rows() -> pd.DataFrame:
    query = f"""
        SELECT unique_id, ds
        FROM {TIMESCALE_CLOSE_MODEL_DATASET_TABLE}
        ORDER BY unique_id, ds;
    """

    try:
        with psycopg2.connect(**_timescale_connection_kwargs()) as connection:
            entity_df = pd.read_sql_query(query, connection)
    except psycopg2.Error as error:
        if "does not exist" in str(error):
            return pd.DataFrame()
        raise

    if entity_df.empty:
        return entity_df

    entity_df["ds"] = pd.to_datetime(entity_df["ds"], utc=True)
    entity_df["ds_key"] = _close_model_ds_key(entity_df["ds"])
    entity_df["series_key"] = _close_model_series_key(
        entity_df["unique_id"],
        entity_df["ds_key"],
    )
    entity_df["event_timestamp"] = entity_df["ds"]
    return entity_df[["series_key", "unique_id", "ds", "event_timestamp"]]


In [ ]:
# serving/feast_store.py
def _read_close_model_online_entity_rows() -> pd.DataFrame:
    query = f"""
        SELECT unique_id, ds
        FROM {TIMESCALE_CLOSE_MODEL_DATASET_TABLE}
        ORDER BY unique_id, ds;
    """

    with psycopg2.connect(**_timescale_connection_kwargs()) as connection:
        entity_df = pd.read_sql_query(query, connection)

    if entity_df.empty:
        return entity_df

    entity_df["ds"] = pd.to_datetime(entity_df["ds"], utc=True)
    entity_df["ds_key"] = _close_model_ds_key(entity_df["ds"])
    entity_df["series_key"] = _close_model_series_key(
        entity_df["unique_id"],
        entity_df["ds_key"],
    )
    return entity_df[["series_key", "unique_id", "ds"]]


In [ ]:
# serving/feast_store.py
def load_stock_close_model_dataset_from_feast() -> pl.DataFrame:
    _apply_close_model_dataset_definition()
    entity_df = _read_close_model_entity_rows()

    if entity_df.empty:
        return pl.DataFrame(
            schema={
                "unique_id": pl.Utf8,
                "ds": pl.Datetime("us"),
                "y": pl.Float64,
                "month_sin_1": pl.Float64,
                "month_cos_1": pl.Float64,
                "day_sin_1": pl.Float64,
                "day_cos_1": pl.Float64,
                "day_of_year_sin_1": pl.Float64,
                "day_of_year_cos_1": pl.Float64,
            }
        )

    store = FeatureStore(repo_path=str(FEATURE_REPO_DIR))
    historical_features = store.get_historical_features(
        entity_df=entity_df[["series_key", "event_timestamp"]],
        features=[
            f"stock_close_model_dataset:{feature_name}"
            for feature_name in ["y", *CLOSE_MODEL_TIME_FEATURE_COLUMNS]
        ],
    ).to_df()

    if "event_timestamp" in historical_features:
        historical_features = historical_features.drop(columns=["event_timestamp"])

    historical_features = historical_features.merge(
        entity_df[["series_key", "unique_id", "ds"]],
        on="series_key",
        how="left",
    )

    return (
        pl.from_pandas(historical_features)
        .select(
            pl.col("unique_id").cast(pl.Utf8),
            pl.col("ds").cast(pl.Datetime("us"), strict=False),
            pl.col("y").cast(pl.Float64, strict=False),
            pl.col("month_sin_1").cast(pl.Float64, strict=False),
            pl.col("month_cos_1").cast(pl.Float64, strict=False),
            pl.col("day_sin_1").cast(pl.Float64, strict=False),
            pl.col("day_cos_1").cast(pl.Float64, strict=False),
            pl.col("day_of_year_sin_1").cast(pl.Float64, strict=False),
            pl.col("day_of_year_cos_1").cast(pl.Float64, strict=False),
        )
        .sort(["unique_id", "ds"])
    )


In [ ]:
# serving/feast_store.py
def _empty_close_model_dataset_frame() -> pl.DataFrame:
    return pl.DataFrame(
        schema={
            "unique_id": pl.Utf8,
            "ds": pl.Datetime("us"),
            "y": pl.Float64,
            "month_sin_1": pl.Float64,
            "month_cos_1": pl.Float64,
            "day_sin_1": pl.Float64,
            "day_cos_1": pl.Float64,
            "day_of_year_sin_1": pl.Float64,
            "day_of_year_cos_1": pl.Float64,
        }
    )


In [ ]:
# serving/feast_store.py
def _online_entity_batches(entity_df: pd.DataFrame):
    for start in range(0, len(entity_df), TIMESCALE_WRITE_BATCH_SIZE):
        batch_df = entity_df.iloc[start : start + TIMESCALE_WRITE_BATCH_SIZE].copy()
        yield batch_df, batch_df[["series_key"]].to_dict("records")


In [ ]:
# serving/feast_store.py
def load_stock_close_model_dataset_from_feast_online() -> pl.DataFrame:
    _apply_close_model_dataset_definition()
    entity_df = _read_close_model_online_entity_rows()
    if entity_df.empty:
        return _empty_close_model_dataset_frame()

    store = FeatureStore(repo_path=str(FEATURE_REPO_DIR))
    feature_refs = [
        f"stock_close_model_dataset:{feature_name}"
        for feature_name in ["y", *CLOSE_MODEL_TIME_FEATURE_COLUMNS]
    ]
    frames = []
    for batch_df, entity_rows in _online_entity_batches(entity_df):
        online_features = store.get_online_features(
            features=feature_refs,
            entity_rows=entity_rows,
        ).to_df()
        if not online_features.empty:
            online_features = online_features.merge(
                batch_df[["series_key", "unique_id", "ds"]],
                on="series_key",
                how="left",
            )
            frames.append(pl.from_pandas(online_features))

    if not frames:
        return _empty_close_model_dataset_frame()

    return (
        pl.concat(frames, how="vertical_relaxed")
        .select(
            pl.col("unique_id").cast(pl.Utf8),
            pl.col("ds").cast(pl.Datetime("us"), strict=False),
            pl.col("y").cast(pl.Float64, strict=False),
            pl.col("month_sin_1").cast(pl.Float64, strict=False),
            pl.col("month_cos_1").cast(pl.Float64, strict=False),
            pl.col("day_sin_1").cast(pl.Float64, strict=False),
            pl.col("day_cos_1").cast(pl.Float64, strict=False),
            pl.col("day_of_year_sin_1").cast(pl.Float64, strict=False),
            pl.col("day_of_year_cos_1").cast(pl.Float64, strict=False),
        )
        .sort(["unique_id", "ds"])
    )


In [ ]:
# serving/feast_store.py
def load_stock_close_model_dataset_from_redis() -> pl.DataFrame:
    return load_stock_close_model_dataset_from_feast_online()


In [ ]:
# serving/feast_store.py
def load_stock_close_model_dataset_from_timescale() -> pl.DataFrame:
    columns = ["unique_id", "ds", "y", *CLOSE_MODEL_TIME_FEATURE_COLUMNS]
    quoted_columns = ", ".join(f'"{column}"' for column in columns)
    query = f"""
        SELECT {quoted_columns}
        FROM {TIMESCALE_CLOSE_MODEL_DATASET_TABLE}
        ORDER BY unique_id, ds;
    """

    rows = []
    try:
        with psycopg2.connect(**_timescale_connection_kwargs()) as connection:
            with connection.cursor() as cursor:
                cursor.execute(query)
                while batch := cursor.fetchmany(TIMESCALE_WRITE_BATCH_SIZE):
                    rows.extend(batch)
    except psycopg2.Error as error:
        if "does not exist" in str(error):
            return _empty_close_model_dataset_frame()
        raise

    if not rows:
        return _empty_close_model_dataset_frame()

    return (
        pl.DataFrame(rows, schema=columns, orient="row")
        .select(
            pl.col("unique_id").cast(pl.Utf8),
            pl.col("ds").cast(pl.Datetime("us"), strict=False),
            pl.col("y").cast(pl.Float64, strict=False),
            pl.col("month_sin_1").cast(pl.Float64, strict=False),
            pl.col("month_cos_1").cast(pl.Float64, strict=False),
            pl.col("day_sin_1").cast(pl.Float64, strict=False),
            pl.col("day_cos_1").cast(pl.Float64, strict=False),
            pl.col("day_of_year_sin_1").cast(pl.Float64, strict=False),
            pl.col("day_of_year_cos_1").cast(pl.Float64, strict=False),
        )
        .sort(["unique_id", "ds"])
    )


In [ ]:
# serving/feast_store.py
def _empty_model_training_dataset_frame(
    feature_columns: list[str],
) -> pl.DataFrame:
    return pl.DataFrame(
        schema={
            "unique_id": pl.Utf8,
            "ds": pl.Datetime("us"),
            "y": pl.Float64,
            **{column: pl.Float64 for column in feature_columns},
        }
    )


In [ ]:
# serving/feast_store.py
def _read_tier2_online_entity_rows() -> pd.DataFrame:
    query = f"""
        SELECT symbol, "date"
        FROM {TIMESCALE_TABLE}
        ORDER BY symbol, "date";
    """

    with psycopg2.connect(**_timescale_connection_kwargs()) as connection:
        entity_df = pd.read_sql_query(query, connection)

    if entity_df.empty:
        return entity_df

    entity_df["date"] = pd.to_datetime(entity_df["date"], utc=True)
    entity_df["date_key"] = _model_feature_date_key(entity_df["date"])
    entity_df["feature_key"] = _model_feature_key(
        entity_df["symbol"],
        entity_df["date_key"],
    )
    return entity_df[["feature_key", "symbol", "date"]]


In [ ]:
# serving/feast_store.py
def _tier2_online_entity_batches(entity_df: pd.DataFrame):
    for start in range(0, len(entity_df), TIMESCALE_WRITE_BATCH_SIZE):
        batch_df = entity_df.iloc[start : start + TIMESCALE_WRITE_BATCH_SIZE].copy()
        yield batch_df, batch_df[["feature_key"]].to_dict("records")


In [ ]:
# serving/feast_store.py
def load_stock_model_training_dataset_from_feast_online(
    feature_columns: list[str],
) -> pl.DataFrame:
    _apply_close_model_dataset_definition()
    _apply_model_feature_definitions()
    close_dataset = load_stock_close_model_dataset_from_feast_online()
    entity_df = _read_tier2_online_entity_rows()

    if entity_df.empty or close_dataset.is_empty():
        return _empty_model_training_dataset_frame(feature_columns)

    store = FeatureStore(repo_path=str(FEATURE_REPO_DIR))
    feature_refs = [
        f"stock_model_tier2_dataset:{feature_name}"
        for feature_name in feature_columns
    ]
    frames = []
    for batch_df, entity_rows in _tier2_online_entity_batches(entity_df):
        online_features = store.get_online_features(
            features=feature_refs,
            entity_rows=entity_rows,
        ).to_df()
        if not online_features.empty:
            online_features = online_features.merge(
                batch_df[["feature_key", "symbol", "date"]],
                on="feature_key",
                how="left",
            )
            frames.append(pl.from_pandas(online_features))

    if not frames:
        return _empty_model_training_dataset_frame(feature_columns)

    model_features = (
        pl.concat(frames, how="vertical_relaxed")
        .select(
            pl.col("symbol").cast(pl.Utf8).alias("unique_id"),
            pl.col("date").cast(pl.Datetime("us"), strict=False).alias("ds"),
            *[
                pl.col(column).cast(pl.Float64, strict=False)
                for column in feature_columns
            ],
        )
        .sort(["unique_id", "ds"])
    )

    return (
        close_dataset.join(
            model_features,
            on=["unique_id", "ds"],
            how="inner",
        )
        .select(["unique_id", "ds", "y", *feature_columns])
        .drop_nulls(["unique_id", "ds", "y", *feature_columns])
        .unique(subset=["unique_id", "ds"], keep="last", maintain_order=True)
        .sort(["unique_id", "ds"])
    )


In [ ]:
# serving/feast_store.py
def load_stock_tier1_model_dataset_from_feast_online() -> pl.DataFrame:
    return load_stock_model_training_dataset_from_feast_online(
        MODEL_TIER_FEATURE_COLUMNS["tier1"]
    )


In [ ]:
# serving/feast_store.py
def load_stock_tier2_model_dataset_from_feast_online() -> pl.DataFrame:
    return load_stock_model_training_dataset_from_feast_online(
        MODEL_TIER_FEATURE_COLUMNS["tier2"]
    )


In [ ]:
# serving/feast_store.py
def load_stock_tier3_model_dataset_from_feast_online() -> pl.DataFrame:
    return load_stock_model_training_dataset_from_feast_online(
        MODEL_TIER_FEATURE_COLUMNS["tier3"]
    )


In [ ]:
# serving/feast_store.py
def get_online_model_features(
    symbols: list[str],
    feature_columns: list[str] | None = None,
) -> pl.DataFrame:
    if not symbols:
        return pl.DataFrame()

    _ensure_feature_repo_on_path()
    store = FeatureStore(repo_path=str(FEATURE_REPO_DIR))
    feature_columns = feature_columns or TIER_2_FEATURE_COLUMNS
    online_features = store.get_online_features(
        features=[
            f"stock_model_features:{feature_name}"
            for feature_name in feature_columns
        ],
        entity_rows=[
            {"symbol": symbol}
            for symbol in sorted(set(symbols))
        ],
    ).to_df()

    return pl.from_pandas(online_features)


In [ ]:
# serving/feast_store.py
def publish_model_features(df: pl.DataFrame) -> dict[str, object]:
    timescale_rows = _write_model_features_to_timescale(df)
    feast_online_rows = _apply_feast_definitions_and_push(df)

    return {
        "timescale_table": TIMESCALE_TABLE,
        "timescale_rows": timescale_rows,
        "feast_online_rows": feast_online_rows,
        "feast_feature_view": "stock_model_features",
        "tier_1_features": TIER_1_FEATURE_COLUMNS,
        "tier_2_time_features": [
            "calendar_gap_days",
            *FOURIER_TIME_ENCODING_COLUMNS,
        ],
        "tier_3_features": TIER_3_FEATURE_COLUMNS,
    }


In [ ]:
# serving/feast_store.py
def publish_conventional_gap_trading_to_store(df: pl.DataFrame) -> dict[str, object]:
    timescale_rows = _write_conventional_gap_trading_to_timescale(df)
    return {
        "timescale_table": TIMESCALE_CONVENTIONAL_GAP_TRADING_TABLE,
        "timescale_rows": timescale_rows,
        "columns": CONVENTIONAL_GAP_TRADING_COLUMNS,
    }


In [ ]:
# serving/feast_store.py
def publish_pecnet_preprocessed_training_data(
    df: pl.DataFrame,
) -> dict[str, object]:
    timescale_rows = _write_pecnet_preprocessed_to_timescale(df)
    feast_online_rows = _apply_pecnet_preprocessed_definition_and_push(df)
    return {
        "timescale_table": TIMESCALE_PECNET_PREPROCESSED_TABLE,
        "timescale_rows": timescale_rows,
        "feast_online_rows": feast_online_rows,
        "feast_feature_view": "pecnet_preprocessed_training_data",
        "columns": PECNET_PREPROCESSED_COLUMNS,
    }


## PECNet Training


In [ ]:
def build_pecnet_spec(
    *,
    enabled: bool = True,
    test_horizon: int = 5,
    feature_columns: list[str] | None = None,
    preprocess_params: dict[str, Any] | None = None,
    hyperparams: dict[str, Any] | None = None,
    selection_params: dict[str, Any] | None = None,
    tier_name: str = "tier1",
) -> dict[str, Any]:
    return {
        "enabled": enabled,
        "tier_name": tier_name,
        "test_horizon": test_horizon,
        "feature_columns": feature_columns or [],
        "preprocess_params": preprocess_params or {},
        "hyperparams": hyperparams or {},
        "selection_params": selection_params or {},
    }


In [ ]:
# ml/pecnet_training.py
def to_pecnet_frame(df: pl.DataFrame) -> pd.DataFrame:
    available_feature_columns = [
        column for column in df.columns if column not in non_feature_columns()
    ]
    return (
        df.select(
            pl.col("unique_id").cast(pl.Utf8),
            pl.col("ds").cast(pl.Datetime("us"), strict=False),
            pl.col("y").cast(pl.Float64, strict=False),
            *[
                pl.col(column).cast(pl.Float64, strict=False)
                for column in available_feature_columns
            ],
        )
        .drop_nulls(["unique_id", "ds", "y", *available_feature_columns])
        .unique(subset=["unique_id", "ds"], keep="last", maintain_order=True)
        .sort(["unique_id", "ds"])
        .to_pandas()
    )


In [ ]:
# ml/pecnet_training.py
def make_pecnet_train_test_split(
    dataset: pl.DataFrame,
    *,
    test_horizon: int,
) -> dict[str, pd.DataFrame]:
    model_df = to_pecnet_frame(dataset)
    train_df, test_df = split_train_test_by_horizon(model_df, test_horizon)
    return {
        "full": model_df,
        "train": train_df,
        "test": test_df,
    }


In [ ]:
# ml/pecnet_training.py; notebook-safe path resolver
def _resolve_pecnetframework_path() -> Path:
    pecnet_source = globals().get("PECNETFRAMEWORK_SOURCE")
    workspace_root = globals().get("WORKSPACE_PROJECT_ROOT")
    candidates = [
        Path("/opt/pecnetframework"),
        Path(os.getenv("PECNETFRAMEWORK_PATH")) if os.getenv("PECNETFRAMEWORK_PATH") else None,
        Path(pecnet_source) if pecnet_source else None,
        Path(workspace_root) / "pecnetframework" if workspace_root else None,
        Path.cwd() / "pecnetframework",
        Path.cwd().parent / "pecnetframework",
        Path("/workspaces/yahooquery_lakehouse_revamp/pecnetframework"),
    ]
    for candidate in candidates:
        if candidate is None:
            continue
        path = Path(candidate).expanduser().resolve()
        if (path / "pecnet").exists():
            return path
    raise FileNotFoundError(
        "pecnetframework klasoru bulunamadi. PECNETFRAMEWORK_PATH env var ile "
        "klasoru goster veya repo root altina pecnetframework koy."
    )


In [ ]:
def _load_pecnet_runtime():
    pecnet_path = _resolve_pecnetframework_path()
    if str(pecnet_path) not in sys.path:
        sys.path.insert(0, str(pecnet_path))

    from pecnet.network import PecnetBuilder  # noqa: PLC0415
    from pecnet.models.BasicNN import BasicNN  # noqa: PLC0415
    from pecnet.preprocessing.DataPreprocessor import DataPreprocessor  # noqa: PLC0415
    from pecnet.utils import FeatureSelector, Utility  # noqa: PLC0415

    import torch  # noqa: PLC0415

    return Utility, PecnetBuilder, DataPreprocessor, BasicNN, FeatureSelector, torch


In [ ]:
# ml/pecnet_training.py
def _safe_name(value: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", value)


In [ ]:
def _configure_torch_threads(torch_module) -> dict[str, int | None]:
    requested_threads = cpu_count_from_env("MODEL_N_JOBS")
    torch_module.set_num_threads(requested_threads)

    interop_threads = min(requested_threads, 4)
    try:
        torch_module.set_num_interop_threads(interop_threads)
    except RuntimeError:
        interop_threads = (
            torch_module.get_num_interop_threads()
            if hasattr(torch_module, "get_num_interop_threads")
            else None
        )

    return {
        "torch_num_threads": int(torch_module.get_num_threads()),
        "torch_num_interop_threads": (
            int(interop_threads) if interop_threads is not None else None
        ),
    }

In [ ]:
# ml/pecnet_training.py
def _ticker_test_ratio(row_count: int, test_horizon: int) -> float:
    if row_count <= test_horizon:
        raise ValueError(
            f"PECNet needs more rows than test_horizon. rows={row_count}, "
            f"test_horizon={test_horizon}"
        )
    return min(max(test_horizon / row_count, 0.01), 0.5)


In [ ]:
# ml/pecnet_training.py
def _preprocess_ticker(
    *,
    ticker_df: pd.DataFrame,
    ticker: str,
    feature_columns: list[str],
    preprocess_params: dict[str, Any],
    test_horizon: int,
    data_preprocessor_cls,
) -> dict[str, Any]:
    dp = data_preprocessor_cls()
    dp.reset()

    ticker_df = ticker_df.sort_values("ds").copy()
    test_ratio = _ticker_test_ratio(len(ticker_df), test_horizon)
    params = {
        **preprocess_params,
        "test_ratio": test_ratio,
    }

    target_series = ticker_df["y"].to_numpy(dtype=float)
    X_train_target, X_test_target, y_train, y_test = dp.preprocess(
        data=target_series,
        **params,
    )

    feature_X_trains = []
    feature_X_tests = []
    available_feature_columns = [
        column for column in feature_columns if column in ticker_df.columns
    ]
    for column in available_feature_columns:
        X_train_feature, X_test_feature, _, _ = dp.preprocess(
            data=ticker_df[column].to_numpy(dtype=float),
            **params,
        )
        feature_X_trains.append(X_train_feature)
        feature_X_tests.append(X_test_feature)

    return {
        "ticker": ticker,
        "target_series": target_series,
        "dates": ticker_df["ds"].reset_index(drop=True),
        "X_train_target": X_train_target,
        "X_test_target": X_test_target,
        "y_train": y_train,
        "y_test": y_test,
        "feature_X_trains": feature_X_trains,
        "feature_X_tests": feature_X_tests,
        "feature_names": available_feature_columns,
        "preprocess_params": params,
        "test_ratio": test_ratio,
    }


In [ ]:
# ml/pecnet_training.py
def _as_2d_float_array(values: Any) -> np.ndarray:
    array = np.asarray(values, dtype=float)
    if array.ndim == 0:
        return array.reshape(1, 1)
    if array.ndim == 1:
        return array.reshape(-1, 1)
    return array.reshape(array.shape[0], -1)


In [ ]:
# ml/pecnet_training.py
def _preprocessed_dates(
    ticker_data: dict[str, Any],
    *,
    split_name: str,
    row_count: int,
) -> pd.Series:
    dates = pd.to_datetime(ticker_data["dates"], utc=True)
    test_count = len(ticker_data["y_test"])
    if split_name == "test":
        return dates.tail(test_count).tail(row_count).reset_index(drop=True)

    train_end = max(len(dates) - test_count, 0)
    return dates.iloc[:train_end].tail(row_count).reset_index(drop=True)


In [ ]:
# ml/pecnet_training.py
def _iter_preprocessed_variable_specs(
    ticker_data: dict[str, Any],
    *,
    split_name: str,
) -> list[tuple[int, str, Any]]:
    if split_name == "train":
        feature_arrays = ticker_data["feature_X_trains"]
        target_array = ticker_data["X_train_target"]
    else:
        feature_arrays = ticker_data["feature_X_tests"]
        target_array = ticker_data["X_test_target"]

    return [
        (0, "target", target_array),
        *[
            (index + 1, feature_name, feature_array)
            for index, (feature_name, feature_array) in enumerate(
                zip(ticker_data["feature_names"], feature_arrays, strict=False)
            )
        ],
    ]


In [ ]:
# ml/pecnet_training.py
def _pecnet_preprocessed_training_frame(
    *,
    ticker_data: dict[str, Any],
    tier_name: str,
) -> pd.DataFrame:
    rows = []
    ticker = str(ticker_data["ticker"])
    tier_safe = _safe_name(tier_name)
    ticker_safe = _safe_name(ticker)
    created_timestamp = pd.Timestamp.now(tz="UTC")

    for split_index, split_name in enumerate(("train", "test")):
        y_values = np.asarray(ticker_data[f"y_{split_name}"], dtype=float).reshape(-1)
        for variable_index, variable_name, values in _iter_preprocessed_variable_specs(
            ticker_data,
            split_name=split_name,
        ):
            matrix = _as_2d_float_array(values)
            sample_count = min(matrix.shape[0], len(y_values))
            dates = _preprocessed_dates(
                ticker_data,
                split_name=split_name,
                row_count=sample_count,
            )
            sample_count = min(sample_count, len(dates))
            if sample_count == 0:
                continue

            matrix = matrix[-sample_count:]
            targets = y_values[-sample_count:]
            variable_safe = _safe_name(str(variable_name))
            for sample_index, event_timestamp in enumerate(dates):
                timestamp = pd.Timestamp(event_timestamp).tz_convert("UTC")
                timestamp_key = timestamp.strftime("%Y%m%dT%H%M%S%fZ")
                for step_index, value in enumerate(matrix[sample_index]):
                    row_key = (
                        f"{tier_safe}|{ticker_safe}|{split_name}|{variable_safe}|"
                        f"{sample_index}|{step_index}|{timestamp_key}"
                    )
                    rows.append(
                        {
                            "row_key": row_key,
                            "tier": tier_name,
                            "symbol": ticker,
                            "event_timestamp": timestamp,
                            "split": split_name,
                            "split_index": split_index,
                            "variable_name": str(variable_name),
                            "variable_index": variable_index,
                            "sample_index": sample_index,
                            "step_index": step_index,
                            "value": float(value),
                            "target_y": float(targets[sample_index]),
                            "created_timestamp": created_timestamp,
                        }
                    )

    return pd.DataFrame(rows)


In [ ]:
# ml/pecnet_training.py
def _log_pecnet_preprocessed_inputs(
    *,
    preprocessed_df: pd.DataFrame,
    tier_name: str,
    ticker: str,
) -> None:
    if preprocessed_df.empty:
        return

    artifact_prefix = (
        f"pecnet/{tier_name}/tickers/{_safe_name(str(ticker))}/preprocessed"
    )
    dataset_prefix = (
        f"stock_close_{tier_name}_{_safe_name(str(ticker))}_pecnet_preprocessed"
    )
    log_mlflow_datasets(
        train_df=preprocessed_df[preprocessed_df["split"] == "train"].copy(),
        test_df=preprocessed_df[preprocessed_df["split"] == "test"].copy(),
        dataset_prefix=dataset_prefix,
        artifact_prefix=artifact_prefix,
    )


In [ ]:
# ml/pecnet_training.py
def _publish_pecnet_preprocessed_inputs(
    preprocessed_df: pd.DataFrame,
) -> dict[str, object]:
    if preprocessed_df.empty:
        return {
            "timescale_rows": 0,
            "feast_online_rows": 0,
        }

    # NOTE: project package import omitted; source appears in notebook cells.

    return publish_pecnet_preprocessed_training_data(
        pl.from_pandas(preprocessed_df)
    )


In [ ]:
def _candidate_variable_inputs(
    ticker_data: dict[str, Any],
) -> tuple[list[str], list[np.ndarray], list[np.ndarray]]:
    return (
        ["target_history", *ticker_data["feature_names"]],
        [ticker_data["X_train_target"], *ticker_data["feature_X_trains"]],
        [ticker_data["X_test_target"], *ticker_data["feature_X_tests"]],
    )


In [ ]:
def _selection_row(
    *,
    ticker: str,
    tier_name: str,
    strategy: str,
    order: int,
    feature_index: int,
    feature_name: str,
    correlation: float | None,
    reference_name: str,
    threshold: float | None,
) -> dict[str, Any]:
    return {
        "ticker": ticker,
        "tier": tier_name,
        "strategy": strategy,
        "selection_order": order,
        "feature_index": feature_index,
        "feature_name": feature_name,
        "reference_name": reference_name,
        "correlation": correlation,
        "abs_correlation": abs(correlation) if correlation is not None else None,
        "threshold": threshold,
    }


In [ ]:
def _log_feature_selection_heatmap(
    selection_df: pd.DataFrame,
    *,
    artifact_file: str,
    title: str,
    index_column: str = "ticker",
) -> None:
    if selection_df.empty or "correlation" not in selection_df.columns:
        return

    correlation_df = selection_df.copy()
    correlation_df["correlation"] = pd.to_numeric(
        correlation_df["correlation"],
        errors="coerce",
    )
    correlation_df = correlation_df[np.isfinite(correlation_df["correlation"])]
    if correlation_df.empty:
        return

    if index_column == "selection_order":
        correlation_df["_heatmap_index"] = (
            "#"
            + correlation_df["selection_order"].astype(int).astype(str)
            + " | "
            + correlation_df["reference_name"].astype(str)
        )
    else:
        correlation_df["_heatmap_index"] = correlation_df[index_column].astype(str)

    heatmap = correlation_df.pivot_table(
        index="_heatmap_index",
        columns="feature_name",
        values="correlation",
        aggfunc="mean",
    )
    if heatmap.empty:
        return

    width = max(8.0, min(24.0, 1.15 * len(heatmap.columns) + 4.0))
    height = max(4.5, min(18.0, 0.55 * len(heatmap.index) + 3.0))
    figure, axis = plt.subplots(figsize=(width, height))
    matrix = heatmap.to_numpy(dtype=float)
    image = axis.imshow(
        np.ma.masked_invalid(matrix),
        cmap="coolwarm",
        vmin=-1.0,
        vmax=1.0,
        aspect="auto",
    )

    axis.set_title(title)
    axis.set_xlabel("Feature")
    axis.set_ylabel("Selection" if index_column == "selection_order" else index_column)
    axis.set_xticks(np.arange(len(heatmap.columns)))
    axis.set_xticklabels(heatmap.columns, rotation=45, ha="right")
    axis.set_yticks(np.arange(len(heatmap.index)))
    axis.set_yticklabels(heatmap.index)

    if matrix.size <= 180:
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                value = matrix[row_index, column_index]
                if np.isfinite(value):
                    axis.text(
                        column_index,
                        row_index,
                        f"{value:.2f}",
                        ha="center",
                        va="center",
                        fontsize=8,
                        color="black",
                    )

    figure.colorbar(image, ax=axis, label="Pearson correlation")
    figure.tight_layout()
    mlflow.log_figure(figure, artifact_file=artifact_file)
    plt.close(figure)

In [ ]:
def _build_all_feature_pecnet_variables(
    *,
    builder,
    ticker_data: dict[str, Any],
    tier_name: str,
) -> tuple[Any, list[np.ndarray], pd.DataFrame]:
    candidate_names, _, candidate_X_test = _candidate_variable_inputs(ticker_data)
    builder.add_variable_network(
        ticker_data["X_train_target"],
        ticker_data["y_train"],
    )
    rows = [
        _selection_row(
            ticker=ticker_data["ticker"],
            tier_name=tier_name,
            strategy="all_features",
            order=1,
            feature_index=0,
            feature_name=candidate_names[0],
            correlation=None,
            reference_name="target_y",
            threshold=None,
        )
    ]
    for feature_order, (feature_name, X_train_feature) in enumerate(
        zip(
            ticker_data["feature_names"],
            ticker_data["feature_X_trains"],
            strict=False,
        ),
        start=2,
    ):
        builder.add_variable_network(
            X_train_feature,
            builder.pecnet.get_target_values_for_current_variable_network(),
        )
        rows.append(
            _selection_row(
                ticker=ticker_data["ticker"],
                tier_name=tier_name,
                strategy="all_features",
                order=feature_order,
                feature_index=feature_order - 1,
                feature_name=feature_name,
                correlation=None,
                reference_name="residual_error",
                threshold=None,
            )
        )

    return builder, candidate_X_test, pd.DataFrame(rows)


In [ ]:
def _build_residual_correlation_pecnet_variables(
    *,
    builder,
    ticker_data: dict[str, Any],
    tier_name: str,
    feature_selector_cls,
    selection_params: dict[str, Any],
) -> tuple[Any, list[np.ndarray], pd.DataFrame]:
    candidate_names, candidate_X_train, candidate_X_test = _candidate_variable_inputs(
        ticker_data
    )
    threshold = float(selection_params.get("correlation_threshold", 0.08))
    max_selected_features = selection_params.get("max_selected_features")
    if max_selected_features is not None:
        max_selected_features = int(max_selected_features)
    force_first = bool(selection_params.get("force_include_best_if_first", True))
    selector = feature_selector_cls(threshold=threshold)
    reference = ticker_data["y_train"]
    rows = []
    initial_network = True

    while True:
        if max_selected_features is not None and len(rows) >= max_selected_features:
            break

        selected_index = selector.select_next(
            candidate_X_train,
            reference,
            force_include_best_if_first=force_first and initial_network,
        )
        if selected_index is None:
            break

        builder.add_variable_network(
            candidate_X_train[selected_index],
            ticker_data["y_train"] if initial_network else reference,
        )
        correlation = selector.get_last_corr_score()
        rows.append(
            _selection_row(
                ticker=ticker_data["ticker"],
                tier_name=tier_name,
                strategy="residual_correlation",
                order=len(rows) + 1,
                feature_index=selected_index,
                feature_name=candidate_names[selected_index],
                correlation=float(correlation) if correlation is not None else None,
                reference_name="target_y" if initial_network else "residual_error",
                threshold=threshold,
            )
        )
        reference = builder.pecnet.get_target_values_for_current_variable_network()
        initial_network = False

    if not rows:
        builder.add_variable_network(
            ticker_data["X_train_target"],
            ticker_data["y_train"],
        )
        rows.append(
            _selection_row(
                ticker=ticker_data["ticker"],
                tier_name=tier_name,
                strategy="residual_correlation_fallback",
                order=1,
                feature_index=0,
                feature_name="target_history",
                correlation=None,
                reference_name="target_y",
                threshold=threshold,
            )
        )

    selected_X_test = [
        candidate_X_test[int(row["feature_index"])]
        for row in rows
    ]
    return builder, selected_X_test, pd.DataFrame(rows)


In [ ]:
def _build_pecnet_variables(
    *,
    builder,
    ticker_data: dict[str, Any],
    tier_name: str,
    feature_selector_cls,
    selection_params: dict[str, Any],
) -> tuple[Any, list[np.ndarray], pd.DataFrame]:
    strategy_by_tier = selection_params.get("strategy_by_tier", {})
    strategy = strategy_by_tier.get(
        tier_name,
        selection_params.get("strategy", "all_features"),
    )
    if strategy == "residual_correlation":
        return _build_residual_correlation_pecnet_variables(
            builder=builder,
            ticker_data=ticker_data,
            tier_name=tier_name,
            feature_selector_cls=feature_selector_cls,
            selection_params=selection_params,
        )

    return _build_all_feature_pecnet_variables(
        builder=builder,
        ticker_data=ticker_data,
        tier_name=tier_name,
    )


In [ ]:
# ml/pecnet_training.py
def _iter_pecnet_basic_models(pecnet) -> list[tuple[str, Any]]:
    models = []
    for variable_index, variable_network in enumerate(pecnet.variable_networks):
        for model_index, model in enumerate(variable_network.models):
            model_name = getattr(
                model,
                "network_name",
                f"Variable_{variable_index}_Network_{model_index}",
            )
            models.append((model_name, model))

    if pecnet.error_network is not None:
        for model_index, model in enumerate(pecnet.error_network.models):
            model_name = getattr(model, "network_name", f"ErrorNetwork_{model_index}")
            models.append((model_name, model))

    if pecnet.final_network is not None:
        for model_index, model in enumerate(pecnet.final_network.models):
            model_name = getattr(model, "network_name", f"FinalNetwork_{model_index}")
            models.append((model_name, model))

    return models


In [ ]:
# ml/pecnet_training.py
def _pecnet_epoch_metrics_frame(
    *,
    pecnet,
    ticker: str,
    tier_name: str,
) -> pd.DataFrame:
    rows = []
    for network_name, model in _iter_pecnet_basic_models(pecnet):
        for epoch, loss in enumerate(getattr(model, "loss_log", []), start=1):
            rows.append(
                {
                    "tier": tier_name,
                    "ticker": ticker,
                    "network": network_name,
                    "epoch": epoch,
                    "train_loss": float(loss),
                }
            )
    return pd.DataFrame(rows)


In [ ]:
def _mlflow_live_pecnet_epoch_logging(
    *,
    basic_nn_cls,
    ticker: str,
    tier_name: str,
):
    original_fit = basic_nn_cls.fit
    global_step = {"value": 0}
    tier_safe = _safe_name(tier_name)
    ticker_safe = _safe_name(ticker)

    def patched_fit(self, input_values, target_values):
        original_loss_log = getattr(self, "loss_log", [])
        network_name = str(getattr(self, "network_name", "Network"))
        network_safe = _safe_name(network_name)

        class MlflowLossLog(list):
            def append(loss_log, value):
                super().append(value)
                epoch = len(loss_log)
                global_step["value"] += 1
                train_loss = float(value)
                mlflow.log_metric(
                    "pecnet.train_loss",
                    train_loss,
                    step=global_step["value"],
                )
                mlflow.log_metric(
                    f"pecnet.{tier_safe}.train_loss",
                    train_loss,
                    step=global_step["value"],
                )
                mlflow.log_metric(
                    f"pecnet.{tier_safe}.{ticker_safe}.train_loss",
                    train_loss,
                    step=global_step["value"],
                )
                mlflow.log_metric(
                    f"pecnet.{tier_safe}.{ticker_safe}.{network_safe}.train_loss",
                    train_loss,
                    step=global_step["value"],
                )
                mlflow.log_metric(
                    f"pecnet.{tier_safe}.{ticker_safe}.{network_safe}.epoch",
                    float(epoch),
                    step=global_step["value"],
                )

        live_loss_log = MlflowLossLog(original_loss_log)
        self.loss_log = live_loss_log
        try:
            return original_fit(self, input_values, target_values)
        finally:
            if isinstance(original_loss_log, list):
                original_loss_log[:] = list(live_loss_log)
            self.loss_log = original_loss_log

    basic_nn_cls.fit = patched_fit
    try:
        yield
    finally:
        basic_nn_cls.fit = original_fit


In [ ]:
def _log_pecnet_epoch_metrics_to_mlflow(
    *,
    pecnet,
    ticker: str,
    tier_name: str,
) -> pd.DataFrame:
    epoch_metrics = _pecnet_epoch_metrics_frame(
        pecnet=pecnet,
        ticker=ticker,
        tier_name=tier_name,
    )
    if epoch_metrics.empty:
        return epoch_metrics

    ticker_safe = _safe_name(ticker)
    mlflow.log_table(
        epoch_metrics,
        f"pecnet/{tier_name}/epoch_metrics/{ticker_safe}.json",
    )
    for network_name, network_metrics in epoch_metrics.groupby("network", sort=False):
        network_safe = _safe_name(str(network_name))
        mlflow.log_metric(
            f"pecnet.{ticker_safe}.{network_safe}.final_train_loss",
            float(network_metrics.iloc[-1]["train_loss"]),
        )
        mlflow.log_metric(
            f"pecnet.{ticker_safe}.{network_safe}.min_train_loss",
            float(network_metrics["train_loss"].min()),
        )

    return epoch_metrics


In [ ]:
# ml/pecnet_training.py
def _save_pecnet_model_file(
    *,
    pecnet,
    torch_module,
    ticker: str,
    tier_name: str,
) -> Path:
    tier_safe = _safe_name(tier_name)
    ticker_safe = _safe_name(ticker)
    model_dir = Path(tempfile.mkdtemp(prefix=f"pecnet_{tier_safe}_{ticker_safe}_"))
    model_path = model_dir / f"pecnet_{tier_safe}_{ticker_safe}.pt"
    torch_module.save(pecnet, model_path)
    return model_path


In [ ]:
def _train_one_ticker(
    *,
    ticker_data: dict[str, Any],
    ticker_train_df: pd.DataFrame,
    ticker_test_df: pd.DataFrame,
    hyperparams: dict[str, Any],
    utility,
    pecnet_builder_cls,
    basic_nn_cls,
    feature_selector_cls,
    torch_module,
    tier_name: str,
    selection_params: dict[str, Any],
) -> tuple[Any, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    ticker = ticker_data["ticker"]
    utility.set_seed(hyperparams.get("seed", 42))
    utility.set_hyperparameters(
        learning_rate=hyperparams["learning_rate"],
        epoch_size=hyperparams["epoch_size"],
        batch_size=hyperparams["batch_size"],
        hidden_units_sizes=hyperparams["hidden_units_sizes"],
    )

    with _mlflow_live_pecnet_epoch_logging(
        basic_nn_cls=basic_nn_cls,
        ticker=ticker,
        tier_name=tier_name,
    ):
        builder = pecnet_builder_cls()
        builder, selected_X_test, selection_df = _build_pecnet_variables(
            builder=builder,
            ticker_data=ticker_data,
            tier_name=tier_name,
            feature_selector_cls=feature_selector_cls,
            selection_params=selection_params,
        )
        pecnet = builder.add_error_network().add_final_network().build()

    predictions = pecnet.predict(
        *selected_X_test,
        test_target=ticker_data["y_test"],
    )

    if torch_module.is_tensor(predictions):
        predictions_array = predictions.detach().cpu().numpy().reshape(-1)
    else:
        predictions_array = np.asarray(predictions, dtype=float).reshape(-1)

    prediction_dates = (
        ticker_test_df[["unique_id", "ds"]]
        .sort_values(["unique_id", "ds"])
        .tail(len(predictions_array))
        .reset_index(drop=True)
    )
    prediction_dates["PECNet"] = predictions_array[-len(prediction_dates) :]
    joined_df = prediction_dates.merge(
        ticker_test_df[["unique_id", "ds", "y"]],
        on=["unique_id", "ds"],
        how="left",
        validate="one_to_one",
    )
    regression_df = _regression_metrics(joined_df)
    long_direction_df = long_only_directional_metrics(joined_df, ticker_train_df)
    epoch_metrics = _log_pecnet_epoch_metrics_to_mlflow(
        pecnet=pecnet,
        ticker=ticker,
        tier_name=tier_name,
    )
    LOGGER.info(
        "Logged PECNet epoch metrics to MLflow | tier=%s ticker=%s rows=%s",
        tier_name,
        ticker,
        len(epoch_metrics),
    )

    return (
        pecnet,
        joined_df,
        pd.concat(
            [
                regression_df.assign(metric_family="regression"),
                long_direction_df.assign(metric_family="long_direction"),
            ],
            ignore_index=True,
        ),
        selection_df,
    )


In [ ]:
def train_pecnet_models_from_split(
    train_test_split: dict[str, pd.DataFrame],
    *,
    model_spec: dict[str, Any],
) -> dict[str, Any]:
    if not model_spec.get("enabled", True) or train_test_split["full"].empty:
        empty = pd.DataFrame()
        return {
            "models": {},
            "train_rows": len(train_test_split["train"]),
            "test_rows": len(train_test_split["test"]),
            "predictions": empty,
            "regression_metrics": empty,
            "long_direction_metrics": empty,
        }

    tier_name = model_spec.get("tier_name", "tier1")
    Utility, PecnetBuilder, DataPreprocessor, BasicNN, FeatureSelector, torch = (
        _load_pecnet_runtime()
    )
    torch_thread_config = _configure_torch_threads(torch)
    configure_mlflow_tracking(experiment_name=tier_experiment_name(tier_name))

    full_df = train_test_split["full"]
    train_df = train_test_split["train"]
    test_df = train_test_split["test"]
    feature_columns = model_spec["feature_columns"]
    preprocess_params = model_spec["preprocess_params"]
    hyperparams = model_spec["hyperparams"]
    selection_params = model_spec.get("selection_params", {})

    all_models = {}
    prediction_frames = []
    regression_frames = []
    long_direction_frames = []
    selection_frames = []
    preprocessed_store_rows = 0

    with mlflow.start_run(run_name=f"stock-close-{tier_name}-pecnet", nested=True):
        mlflow.log_params(
            {
                "tier_name": tier_name,
                "test_horizon": model_spec["test_horizon"],
                "feature_columns": ",".join(feature_columns),
                        "pecnet.selection_strategy": (
                    selection_params.get("strategy_by_tier", {}).get(
                        tier_name,
                        selection_params.get("strategy", "all_features"),
                    )
                ),
                "pecnet.correlation_threshold": selection_params.get(
                    "correlation_threshold",
                    "",
                )
                or "",
                "pecnet.max_selected_features": selection_params.get(
                    "max_selected_features",
                    "",
                )
                or "",
                "pecnet.torch_num_threads": torch_thread_config["torch_num_threads"],
                "pecnet.torch_num_interop_threads": (
                    torch_thread_config["torch_num_interop_threads"] or ""
                ),
                **{
                    f"pecnet.{key}": value
                    for key, value in hyperparams.items()
                    if not isinstance(value, (list, dict, tuple))
                },
            }
        )
        mlflow.log_dict(preprocess_params, f"pecnet/{tier_name}/preprocess_params.json")
        mlflow.log_dict(hyperparams, f"pecnet/{tier_name}/hyperparams.json")
        mlflow.log_dict(selection_params, f"pecnet/{tier_name}/selection_params.json")
        log_mlflow_datasets(
            train_df=train_df,
            test_df=test_df,
            dataset_prefix=f"stock_close_{tier_name}_pecnet",
            artifact_prefix=f"pecnet/{tier_name}",
        )

        for ticker, ticker_df in full_df.groupby("unique_id", observed=True):
            ticker_train_df = train_df[train_df["unique_id"] == ticker].copy()
            ticker_test_df = test_df[test_df["unique_id"] == ticker].copy()
            if ticker_train_df.empty or ticker_test_df.empty:
                continue

            with mlflow.start_run(
                run_name=f"pecnet-{tier_name}-{_safe_name(str(ticker))}",
                nested=True,
            ):
                log_mlflow_datasets(
                    train_df=ticker_train_df,
                    test_df=ticker_test_df,
                    dataset_prefix=(
                        f"stock_close_{tier_name}_{_safe_name(str(ticker))}_pecnet"
                    ),
                    artifact_prefix=(
                        f"pecnet/{tier_name}/tickers/{_safe_name(str(ticker))}"
                    ),
                )
                ticker_data = _preprocess_ticker(
                    ticker_df=ticker_df,
                    ticker=str(ticker),
                    feature_columns=feature_columns,
                    preprocess_params=preprocess_params,
                    test_horizon=model_spec["test_horizon"],
                    data_preprocessor_cls=DataPreprocessor,
                )
                preprocessed_df = _pecnet_preprocessed_training_frame(
                    ticker_data=ticker_data,
                    tier_name=tier_name,
                )
                _log_pecnet_preprocessed_inputs(
                    preprocessed_df=preprocessed_df,
                    tier_name=tier_name,
                    ticker=str(ticker),
                )
                preprocessed_store_metadata = _publish_pecnet_preprocessed_inputs(
                    preprocessed_df
                )
                preprocessed_store_rows += int(
                    preprocessed_store_metadata.get("timescale_rows", 0)
                )
                mlflow.log_dict(
                    preprocessed_store_metadata,
                    (
                        f"pecnet/{tier_name}/tickers/{_safe_name(str(ticker))}/"
                        "preprocessed/store_metadata.json"
                    ),
                )
                pecnet, joined_df, combined_metrics, selection_df = _train_one_ticker(
                    ticker_data=ticker_data,
                    ticker_train_df=ticker_train_df,
                    ticker_test_df=ticker_test_df,
                    hyperparams=hyperparams,
                    utility=Utility,
                    pecnet_builder_cls=PecnetBuilder,
                    basic_nn_cls=BasicNN,
                    feature_selector_cls=FeatureSelector,
                    torch_module=torch,
                                tier_name=tier_name,
                    selection_params=selection_params,
                )

                regression_df = combined_metrics[
                    combined_metrics["metric_family"] == "regression"
                ].drop(columns=["metric_family"])
                long_direction_df = combined_metrics[
                    combined_metrics["metric_family"] == "long_direction"
                ].drop(columns=["metric_family"])

                all_models[str(ticker)] = pecnet
                prediction_frames.append(joined_df)
                regression_frames.append(regression_df)
                long_direction_frames.append(long_direction_df)
                if not selection_df.empty:
                    selection_frames.append(selection_df)

                mlflow.log_table(joined_df, f"pecnet/{tier_name}/predictions/{ticker}.json")
                mlflow.log_table(
                    regression_df,
                    f"pecnet/{tier_name}/evaluation/{ticker}_regression.json",
                )
                mlflow.log_table(
                    long_direction_df,
                    f"pecnet/{tier_name}/evaluation/{ticker}_long_direction.json",
                )
                ticker_safe = _safe_name(str(ticker))
                if not selection_df.empty:
                    mlflow.log_table(
                        selection_df,
                        f"pecnet/{tier_name}/feature_selection/{ticker_safe}.json",
                    )
                    _log_feature_selection_heatmap(
                        selection_df,
                        artifact_file=(
                            f"pecnet/{tier_name}/feature_selection/"
                            f"{ticker_safe}_correlation_heatmap.png"
                        ),
                        title=(
                            f"PECNet {tier_name} {ticker_safe} "
                            "selected feature correlations"
                        ),
                        index_column="selection_order",
                    )
                    for _, row in selection_df.iterrows():
                        correlation = row.get("correlation")
                        if pd.isna(correlation):
                            continue
                        mlflow.log_metric(
                            (
                                f"pecnet.{ticker_safe}.selection."
                                f"{int(row['selection_order'])}.correlation"
                            ),
                            float(correlation),
                        )
                        mlflow.log_metric(
                            (
                                f"pecnet.{ticker_safe}.selection."
                                f"{int(row['selection_order'])}.abs_correlation"
                            ),
                            float(row["abs_correlation"]),
                        )
                for _, row in regression_df.iterrows():
                    model_safe = _safe_name(str(row["model"]))
                    for metric_name in ["mae", "rmse", "mape", "r2"]:
                        if pd.isna(row.get(metric_name)):
                            continue
                        mlflow.log_metric(
                            (
                                f"pecnet.{ticker_safe}.{model_safe}."
                                f"test.{metric_name}"
                            ),
                            float(row[metric_name]),
                        )

                for _, row in long_direction_df.iterrows():
                    if row[
                        ["long_accuracy", "long_precision", "long_recall"]
                    ].isna().any():
                        continue
                    model_safe = _safe_name(str(row["model"]))
                    for metric_name in [
                        "long_accuracy",
                        "long_precision",
                        "long_recall",
                        "long_signal_rate",
                    ]:
                        if pd.isna(row.get(metric_name)):
                            continue
                        mlflow.log_metric(
                            f"pecnet.{ticker_safe}.{model_safe}.{metric_name}",
                            float(row[metric_name]),
                        )
                log_forecast_plots(
                    train_df=ticker_train_df,
                    joined_df=joined_df,
                    levels=None,
                    artifact_prefix=f"pecnet/{tier_name}/plots/{ticker_safe}",
                )

                figure, comparison_df = pecnet_prediction_figure(
                    predictions=joined_df["PECNet"].to_numpy(),
                    actual=joined_df["y"].to_numpy(),
                    dates=joined_df["ds"],
                    model_name=f"PECNet {ticker}",
                )
                mlflow.log_figure(
                    figure,
                    artifact_file=(
                        f"pecnet/{tier_name}/plots/"
                        f"{ticker_safe}/comparison.png"
                    ),
                )
                plt.close(figure)
                mlflow.log_table(
                    comparison_df,
                    f"pecnet/{tier_name}/predictions/"
                    f"{ticker_safe}_comparison.json",
                )

                model_dir = Path(tempfile.mkdtemp(prefix="pecnet_"))
                model_path = model_dir / f"pecnet_{ticker_safe}.pt"
                try:
                    torch.save(pecnet, model_path)
                    mlflow.log_artifact(
                        str(model_path),
                        artifact_path=f"pecnet/{tier_name}/models/{ticker_safe}",
                    )
                except Exception:
                    LOGGER.warning("Could not serialize PECNet model.", exc_info=True)

        predictions = (
            pd.concat(prediction_frames, ignore_index=True)
            if prediction_frames
            else pd.DataFrame()
        )
        regression_metrics = (
            pd.concat(regression_frames, ignore_index=True)
            if regression_frames
            else pd.DataFrame()
        )
        long_direction_metrics = (
            pd.concat(long_direction_frames, ignore_index=True)
            if long_direction_frames
            else pd.DataFrame()
        )
        feature_selection = (
            pd.concat(selection_frames, ignore_index=True)
            if selection_frames
            else pd.DataFrame()
        )
        mlflow.log_table(predictions, f"pecnet/{tier_name}/predictions/all_predictions.json")
        mlflow.log_table(
            regression_metrics,
            f"pecnet/{tier_name}/evaluation/all_regression_metrics.json",
        )
        mlflow.log_table(
            long_direction_metrics,
            f"pecnet/{tier_name}/evaluation/all_long_direction_metrics.json",
        )
        mlflow.log_table(
            feature_selection,
            f"pecnet/{tier_name}/feature_selection/all_feature_selection.json",
        )
        _log_feature_selection_heatmap(
            feature_selection,
            artifact_file=(
                f"pecnet/{tier_name}/feature_selection/"
                "all_correlation_heatmap.png"
            ),
            title=f"PECNet {tier_name} selected feature correlations",
            index_column="ticker",
        )

    return {
        "models": all_models,
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "predictions": predictions,
        "regression_metrics": regression_metrics,
        "long_direction_metrics": long_direction_metrics,
        "feature_selection": feature_selection,
        "preprocessed_store_rows": preprocessed_store_rows,
    }

## Pipeline Nodes


In [ ]:
# nodes.py
def _log_step(step_name: str, **metadata: Any) -> None:
    print(f"\n[{step_name}]")
    for key, value in metadata.items():
        print(f"  {key}: {value}")


In [ ]:
# nodes.py
def _bool_from_env(env_name: str, default: bool) -> bool:
    value = os.getenv(env_name)
    if value is None:
        return default
    return value.lower() in {"1", "true", "yes"}


In [ ]:
# nodes.py
def _int_from_env(env_name: str, default: int) -> int:
    value = os.getenv(env_name)
    if value is None:
        return default
    return int(value)


In [ ]:
# nodes.py
def _int_from_envs(env_names: tuple[str, ...], default: int) -> tuple[int, str]:
    for env_name in env_names:
        value = os.getenv(env_name)
        if value is not None:
            return int(value), f"env:{env_name}"
    return int(default), "parameters"


In [ ]:
# nodes.py
def _set_env_default(env_name: str, value: Any) -> None:
    os.environ.setdefault(env_name, str(value))


In [ ]:
# nodes.py; notebook-safe credentials lookup
def _load_local_credentials() -> dict[str, Any]:
    env_credentials_path = os.getenv("KEDRO_CREDENTIALS_PATH")
    kedro_project = globals().get("KEDRO_PROJECT")
    if not kedro_project and os.getenv("KEDRO_PROJECT_DIR"):
        kedro_project = Path(os.getenv("KEDRO_PROJECT_DIR"))
    workspace_root = globals().get("WORKSPACE_PROJECT_ROOT")

    candidates = [
        Path(env_credentials_path) if env_credentials_path else None,
        Path.cwd() / "conf" / "local" / "credentials.yml",
        Path(kedro_project) / "conf" / "local" / "credentials.yml" if kedro_project else None,
        Path(workspace_root) / "kedro_project" / "conf" / "local" / "credentials.yml" if workspace_root else None,
        Path("/opt/kedro_project/conf/local/credentials.yml"),
        Path("/workspaces/yahooquery_lakehouse_revamp/kedro_project/conf/local/credentials.yml"),
    ]
    for path in candidates:
        if path is None:
            continue
        path = Path(path).expanduser()
        if not path.exists():
            continue
        try:
            with path.open() as file:
                return yaml.safe_load(file) or {}
        except OSError as exc:
            if getattr(exc, "errno", None) == errno.EDEADLK:
                continue
            raise
    return {}


In [ ]:
def start_config(
    data_preprocessing_parameters: dict[str, Any] | None = None,
    machine_learning_parameters: dict[str, Any] | None = None,
) -> dict[str, Any]:
    data_preprocessing_parameters = data_preprocessing_parameters or {}
    machine_learning_parameters = machine_learning_parameters or {}
    delta_lake = data_preprocessing_parameters.get("delta_lake", {})
    feature_engineering = data_preprocessing_parameters.get("feature_engineering", {})
    conventional_gap_trading = data_preprocessing_parameters.get(
        "conventional_gap_trading",
        {},
    )
    time_encoding = data_preprocessing_parameters.get("time_encoding", {})
    columns_config = resolve_column_config(
        data_preprocessing_parameters.get("columns", {})
    )
    training = machine_learning_parameters.get("training", {})
    mlflow_config = machine_learning_parameters.get("mlflow", {})
    mlforecast_config = machine_learning_parameters.get("mlforecast", {})
    statsforecast_config = machine_learning_parameters.get("statsforecast", {})
    pecnet_config = machine_learning_parameters.get("pecnet", {})
    validation_horizon, validation_horizon_source = _int_from_envs(
        ("MODEL_VALIDATION_HORIZON", "MLFORECAST_VALIDATION_HORIZON"),
        training.get(
            "validation_horizon",
            mlforecast_config.get("validation_horizon", 1),
        ),
    )
    test_horizon, test_horizon_source = _int_from_envs(
        ("MODEL_TEST_HORIZON", "MLFORECAST_TEST_HORIZON"),
        training.get(
            "test_horizon",
            mlforecast_config.get("test_horizon", 3),
        ),
    )
    runtime_config = machine_learning_parameters.get("runtime", {})

    config = {
        "bucket": os.getenv(
            "DELTA_LAKE_S3_BUCKET",
            delta_lake.get("bucket", "delta-lake-bucket"),
        ),
        "training_data_source": os.getenv(
            "MODEL_TRAINING_DATA_SOURCE",
            training.get("data_source", "feast_online"),
        ),
        "publish_indicator_features": _bool_from_env(
            "PUBLISH_INDICATOR_FEATURES",
            feature_engineering.get("publish_indicator_features", True),
        ),
        "publish_conventional_gap_trading": _bool_from_env(
            "PUBLISH_CONVENTIONAL_GAP_TRADING",
            conventional_gap_trading.get("publish_to_timescale", True),
        ),
        "mlflow_tracking_uri": os.getenv(
            "MLFLOW_TRACKING_URI",
            mlflow_config.get("tracking_uri", "http://host.docker.internal:5001"),
        ),
        "mlflow_experiment_name": os.getenv(
            "MLFLOW_EXPERIMENT_NAME",
            mlflow_config.get("experiment_name", "stock_close_training"),
        ),
        "mlflow_tier_experiment_prefix": os.getenv(
            "MLFLOW_TIER_EXPERIMENT_PREFIX",
            mlflow_config.get("tier_experiment_prefix", "stock_close"),
        ),
        "mlflow_request_timeout": _int_from_env(
            "MLFLOW_HTTP_REQUEST_TIMEOUT",
            mlflow_config.get("request_timeout", 10),
        ),
        "mlflow_request_max_retries": _int_from_env(
            "MLFLOW_HTTP_REQUEST_MAX_RETRIES",
            mlflow_config.get("request_max_retries", 1),
        ),
        "mlflow_request_backoff_factor": _int_from_env(
            "MLFLOW_HTTP_REQUEST_BACKOFF_FACTOR",
            mlflow_config.get("request_backoff_factor", 1),
        ),
        "freq": os.getenv("MLFORECAST_FREQ", mlforecast_config.get("freq", "B")),
        "validation_horizon": validation_horizon,
        "validation_horizon_source": validation_horizon_source,
        "test_horizon": test_horizon,
        "test_horizon_source": test_horizon_source,
        "n_windows": _int_from_env(
            "MLFORECAST_N_WINDOWS",
            mlforecast_config.get("n_windows", 1),
        ),
        "n_trials": _int_from_env(
            "MLFORECAST_N_TRIALS",
            mlforecast_config.get("n_trials", 1),
        ),
        "verbose": _bool_from_env(
            "MLFORECAST_VERBOSE",
            mlforecast_config.get("verbose", True),
        ),
        "estimator_verbose": _bool_from_env(
            "MODEL_ESTIMATOR_VERBOSE",
            mlforecast_config.get("estimator_verbose", False),
        ),
        "mlforecast_models": mlforecast_config.get("models"),
        "mlforecast_num_threads": _int_from_env(
            "MLFORECAST_NUM_THREADS",
            runtime_config.get("mlforecast_num_threads", 1),
        ),
        "model_n_jobs": _int_from_env(
            "MODEL_N_JOBS",
            runtime_config.get("model_n_jobs", 1),
        ),
        "model_max_estimators": _int_from_env(
            "MODEL_MAX_ESTIMATORS",
            runtime_config.get("model_max_estimators", 300),
        ),
        "statsforecast": {
            "enabled": _bool_from_env(
                "STATSFORECAST_ENABLED",
                statsforecast_config.get("enabled", True),
            ),
            "freq": os.getenv(
                "STATSFORECAST_FREQ",
                statsforecast_config.get("freq", mlforecast_config.get("freq", "B")),
            ),
            "seasonal_length": _int_from_env(
                "STATSFORECAST_SEASONAL_LENGTH",
                statsforecast_config.get("seasonal_length", 5),
            ),
            "conformal_n_windows": _int_from_env(
                "STATSFORECAST_CONFORMAL_N_WINDOWS",
                statsforecast_config.get(
                    "conformal_n_windows",
                    mlforecast_config.get("n_windows", 3),
                ),
            ),
            "level": statsforecast_config.get("level", [80, 95]),
            "models": statsforecast_config.get("models"),
            "verbose": _bool_from_env(
                "STATSFORECAST_VERBOSE",
                statsforecast_config.get("verbose", True),
            ),
        },
        "pecnet": {
            "enabled": _bool_from_env(
                "PECNET_ENABLED",
                pecnet_config.get("enabled", True),
            ),
            "feature_columns_by_tier": pecnet_config.get("feature_columns_by_tier"),
            "selection_params": pecnet_config.get("selection_params", {}),
            "preprocess_params": pecnet_config.get("preprocess_params"),
            "hyperparams": pecnet_config.get("hyperparams"),
        },
        "time_encoding": time_encoding,
        "columns": columns_config,
        "model_tier_feature_columns": {
            tier_name.removesuffix("_features").replace("_", ""): feature_columns
            for tier_name, feature_columns in columns_config.items()
            if tier_name.startswith("tier_") and tier_name.endswith("_features")
        },
    }
    config["indicator_features_path"] = feature_engineering.get(
        "indicator_features_path",
        stock_price_indicator_features_path(config["bucket"]),
    )
    os.environ["MLFLOW_TRACKING_URI"] = config["mlflow_tracking_uri"]
    os.environ["MLFLOW_EXPERIMENT_NAME"] = config["mlflow_experiment_name"]
    os.environ["MLFLOW_TIER_EXPERIMENT_PREFIX"] = config[
        "mlflow_tier_experiment_prefix"
    ]
    _set_env_default(
        "MLFLOW_HTTP_REQUEST_TIMEOUT",
        config["mlflow_request_timeout"],
    )
    _set_env_default(
        "MLFLOW_HTTP_REQUEST_MAX_RETRIES",
        config["mlflow_request_max_retries"],
    )
    _set_env_default(
        "MLFLOW_HTTP_REQUEST_BACKOFF_FACTOR",
        config["mlflow_request_backoff_factor"],
    )
    _set_env_default("MLFORECAST_NUM_THREADS", config["mlforecast_num_threads"])
    _set_env_default("MODEL_N_JOBS", config["model_n_jobs"])
    _set_env_default("MODEL_MAX_ESTIMATORS", config["model_max_estimators"])
    os.environ["MODEL_ESTIMATOR_VERBOSE"] = (
        "1" if config["estimator_verbose"] else "0"
    )
    _log_step("start", **config)
    return config


In [ ]:
# nodes.py
def _feature_columns_for_tier(
    run_config: dict[str, Any],
    tier_name: str,
) -> list[str]:
    try:
        return run_config["model_tier_feature_columns"][tier_name]
    except KeyError as exc:
        raise ValueError(
            f"Unknown model tier {tier_name!r}. "
            f"Expected one of {sorted(run_config['model_tier_feature_columns'])}."
        ) from exc


In [ ]:
# nodes.py
def prepare_close_model_dataset(
    run_config: dict[str, Any],
) -> tuple[pl.DataFrame, dict[str, Any]]:
    silver_stock_prices = read_silver_stock_prices(run_config["bucket"])
    model_dataset = build_stock_close_model_dataset(
        silver_stock_prices,
        run_config["columns"],
        run_config["time_encoding"],
    )
    metadata = {
        "silver_rows": len(silver_stock_prices),
        "close_model_rows": len(model_dataset),
        "symbols": model_dataset["unique_id"].n_unique(),
        "min_ds": model_dataset["ds"].min(),
        "max_ds": model_dataset["ds"].max(),
    }
    _log_step("prepare_close_model_dataset", **metadata)
    return model_dataset, metadata


In [ ]:
# nodes.py
def publish_close_model_dataset(stock_close_model_dataset: pl.DataFrame) -> dict[str, Any]:
    metadata = publish_close_model_dataset_to_store(stock_close_model_dataset)
    _log_step("publish_close_model_dataset", **metadata)
    return metadata


In [ ]:
# nodes.py
def prepare_indicator_features(
    run_config: dict[str, Any],
) -> tuple[pl.DataFrame, dict[str, Any]]:
    if not run_config["publish_indicator_features"]:
        indicator_features = pl.DataFrame()
        metadata = {
            "publish_indicator_features": False,
            "indicator_feature_rows": 0,
            "indicator_features_path": run_config["indicator_features_path"],
        }
        _log_step("prepare_indicator_features", **metadata)
        return indicator_features, metadata

    silver_stock_prices = read_silver_stock_prices(run_config["bucket"])
    indicator_features = build_stock_price_indicator_features(
        silver_stock_prices,
        run_config["columns"],
        run_config["time_encoding"],
    )
    write_delta_table(run_config["indicator_features_path"], indicator_features)
    metadata = {
        "publish_indicator_features": True,
        "silver_rows": len(silver_stock_prices),
        "indicator_feature_rows": len(indicator_features),
        "indicator_features_path": run_config["indicator_features_path"],
    }
    _log_step("prepare_indicator_features", **metadata)
    return indicator_features, metadata


In [ ]:
# nodes.py
def load_indicator_features(
    run_config: dict[str, Any],
) -> tuple[pl.DataFrame, dict[str, Any]]:
    indicator_features = read_delta_table(run_config["indicator_features_path"])
    metadata = {
        "indicator_feature_rows": len(indicator_features),
        "indicator_features_path": run_config["indicator_features_path"],
    }
    _log_step("load_indicator_features", **metadata)
    return indicator_features, metadata


In [ ]:
# nodes.py
def publish_indicator_model_features(
    stock_price_indicator_features: pl.DataFrame,
    indicator_feature_metadata: dict[str, Any],
) -> dict[str, Any]:
    if stock_price_indicator_features.is_empty():
        metadata = {
            "skipped": True,
            "reason": "stock_price_indicator_features is empty",
        }
        _log_step("publish_indicator_model_features", **metadata)
        return metadata

    metadata = publish_model_features(stock_price_indicator_features)
    metadata["indicator_feature_rows"] = indicator_feature_metadata.get(
        "indicator_feature_rows",
        len(stock_price_indicator_features),
    )
    _log_step("publish_indicator_model_features", **metadata)
    return metadata


In [ ]:
# nodes.py
def prepare_conventional_gap_trading(
    stock_price_indicator_features: pl.DataFrame,
    run_config: dict[str, Any],
) -> tuple[pl.DataFrame, dict[str, Any]]:
    if not run_config["publish_conventional_gap_trading"]:
        conventional_gap_trading = pl.DataFrame()
        metadata = {
            "publish_conventional_gap_trading": False,
            "conventional_gap_trading_rows": 0,
        }
        _log_step("prepare_conventional_gap_trading", **metadata)
        return conventional_gap_trading, metadata

    if stock_price_indicator_features.is_empty():
        conventional_gap_trading = pl.DataFrame()
        metadata = {
            "publish_conventional_gap_trading": True,
            "indicator_feature_rows": 0,
            "conventional_gap_trading_rows": 0,
            "signals": {},
        }
        _log_step("prepare_conventional_gap_trading", **metadata)
        return conventional_gap_trading, metadata

    conventional_gap_trading = build_conventional_gap_trading_features(
        stock_price_indicator_features,
        run_config["columns"],
    )
    metadata = {
        "publish_conventional_gap_trading": True,
        "indicator_feature_rows": len(stock_price_indicator_features),
        "conventional_gap_trading_rows": len(conventional_gap_trading),
        "signals": (
            conventional_gap_trading.get_column("Gap_Type")
            .value_counts()
            .to_dict(as_series=False)
            if "Gap_Type" in conventional_gap_trading.columns
            else {}
        ),
    }
    _log_step("prepare_conventional_gap_trading", **metadata)
    return conventional_gap_trading, metadata


In [ ]:
# nodes.py
def publish_conventional_gap_trading(
    conventional_gap_trading: pl.DataFrame,
    conventional_gap_trading_metadata: dict[str, Any],
) -> dict[str, Any]:
    if not conventional_gap_trading_metadata["publish_conventional_gap_trading"]:
        metadata = {
            "skipped": True,
            "reason": "publish_conventional_gap_trading is false",
        }
        _log_step("publish_conventional_gap_trading", **metadata)
        return metadata

    metadata = publish_conventional_gap_trading_to_store(conventional_gap_trading)
    _log_step("publish_conventional_gap_trading", **metadata)
    return metadata


In [ ]:
# nodes.py
def load_model_training_dataset(
    run_config: dict[str, Any],
    *,
    tier_name: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    feature_columns = _feature_columns_for_tier(run_config, tier_name)
    training_dataset = load_stock_model_training_dataset_from_feast_online(
        feature_columns
    )
    metadata = {
        "tier": tier_name,
        "training_data_source": "feast_online",
        "training_rows": len(training_dataset),
        "symbols": training_dataset["unique_id"].n_unique()
        if "unique_id" in training_dataset.columns
        else 0,
        "feature_columns": feature_columns,
    }
    _log_step(f"load_{tier_name}_training_dataset", **metadata)
    return training_dataset, metadata


In [ ]:
# nodes.py
def load_model_training_dataset_after_feature_publish(
    run_config: dict[str, Any],
    model_feature_publish_metadata: dict[str, Any],
    *,
    tier_name: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    return load_model_training_dataset(run_config, tier_name=tier_name)


In [ ]:
# nodes.py
def train_test_split_for_tier(
    stock_close_training_dataset: pl.DataFrame,
    run_config: dict[str, Any],
    *,
    tier_name: str,
) -> tuple[dict[str, pd.DataFrame], dict[str, Any]]:
    split = make_train_test_split(
        stock_close_training_dataset,
        test_horizon=run_config["test_horizon"],
    )
    metadata = {
        "tier": tier_name,
        "train_rows": len(split["train"]),
        "test_rows": len(split["test"]),
        "test_horizon": run_config["test_horizon"],
    }
    _log_step(f"{tier_name}_train_test_split", **metadata)
    return split, metadata


In [ ]:
# nodes.py
def build_model_spec(
    run_config: dict[str, Any],
    *,
    tier_name: str = "tier1",
) -> dict[str, Any]:
    model_spec = build_auto_mlforecast_spec(
        freq=run_config["freq"],
        validation_horizon=run_config["validation_horizon"],
        test_horizon=run_config["test_horizon"],
        n_windows=run_config["n_windows"],
        n_trials=run_config["n_trials"],
        verbose=run_config["verbose"],
        models=run_config["mlforecast_models"],
        tier_name=tier_name,
    )
    _log_step(f"build_{tier_name}_mlforecast_model_spec", **model_spec)
    return model_spec


In [ ]:
# nodes.py
def train_models(
    stock_close_train_test_split: dict[str, pd.DataFrame],
    stock_close_model_spec: dict[str, Any],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    result = train_auto_mlforecast_models_from_split(
        stock_close_train_test_split,
        model_spec=stock_close_model_spec,
    )
    regression_metrics = result["regression_metrics"]
    long_direction_metrics = result["long_direction_metrics"]
    predictions = result["predictions"]

    best_model = None
    if not regression_metrics.empty:
        best_model = (
            regression_metrics.sort_values("rmse", ascending=True).iloc[0]["model"]
        )

    metadata = {
        "tier": stock_close_model_spec.get("tier_name", "tier1"),
        "train_rows": result["train_rows"],
        "test_rows": result["test_rows"],
        "best_model": best_model,
        "regression_metric_rows": len(regression_metrics),
        "long_direction_metric_rows": len(long_direction_metrics),
        "prediction_rows": len(predictions),
    }
    _log_step("train_models", **metadata)
    return regression_metrics, long_direction_metrics, predictions, metadata


In [ ]:
# nodes.py
def build_statsforecast_model_spec_for_tier(
    run_config: dict[str, Any],
    *,
    tier_name: str,
) -> dict[str, Any]:
    stats_config = run_config["statsforecast"]
    model_spec = build_statsforecast_spec(
        freq=stats_config["freq"],
        seasonal_length=stats_config["seasonal_length"],
        validation_horizon=run_config["validation_horizon"],
        test_horizon=run_config["test_horizon"],
        conformal_n_windows=stats_config["conformal_n_windows"],
        level=stats_config["level"],
        models=stats_config["models"],
        verbose=stats_config["verbose"],
        tier_name=tier_name,
    )
    model_spec["enabled"] = stats_config["enabled"]
    _log_step(f"build_{tier_name}_statsforecast_model_spec", **model_spec)
    return model_spec


In [ ]:
# nodes.py
def train_statsforecast_models(
    stock_close_train_test_split: dict[str, pd.DataFrame],
    stock_close_statsforecast_model_spec: dict[str, Any],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    if not stock_close_statsforecast_model_spec.get("enabled", True):
        empty = pd.DataFrame()
        metadata = {"skipped": True, "reason": "statsforecast disabled"}
        _log_step("train_statsforecast_models", **metadata)
        return empty, empty, empty, metadata

    result = train_statsforecast_models_from_split(
        stock_close_train_test_split,
        model_spec=stock_close_statsforecast_model_spec,
    )
    regression_metrics = result["regression_metrics"]
    long_direction_metrics = result["long_direction_metrics"]
    predictions = result["predictions"]
    best_model = None
    if not regression_metrics.empty:
        best_model = (
            regression_metrics.sort_values("rmse", ascending=True).iloc[0]["model"]
        )

    metadata = {
        "tier": stock_close_statsforecast_model_spec.get("tier_name", "tier1"),
        "train_rows": result["train_rows"],
        "test_rows": result["test_rows"],
        "best_model": best_model,
        "regression_metric_rows": len(regression_metrics),
        "long_direction_metric_rows": len(long_direction_metrics),
        "prediction_rows": len(predictions),
    }
    _log_step("train_statsforecast_models", **metadata)
    return regression_metrics, long_direction_metrics, predictions, metadata


In [ ]:
def build_pecnet_model_spec_for_tier(
    run_config: dict[str, Any],
    *,
    tier_name: str,
) -> dict[str, Any]:
    pecnet_config = run_config["pecnet"]
    feature_columns_by_tier = pecnet_config.get("feature_columns_by_tier") or {}
    feature_columns = feature_columns_by_tier.get(
        tier_name,
        _feature_columns_for_tier(run_config, tier_name),
    )
    model_spec = build_pecnet_spec(
        enabled=pecnet_config["enabled"],
        test_horizon=run_config["test_horizon"],
        feature_columns=feature_columns,
        preprocess_params=pecnet_config["preprocess_params"],
        hyperparams=pecnet_config["hyperparams"],
        selection_params=pecnet_config.get("selection_params", {}),
        tier_name=tier_name,
    )
    _log_step(f"build_{tier_name}_pecnet_model_spec", **model_spec)
    return model_spec


In [ ]:
def train_pecnet_models(
    stock_close_pecnet_train_test_split: dict[str, pd.DataFrame],
    stock_close_pecnet_model_spec: dict[str, Any],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    result = train_pecnet_models_from_split(
        stock_close_pecnet_train_test_split,
        model_spec=stock_close_pecnet_model_spec,
    )
    regression_metrics = result["regression_metrics"]
    long_direction_metrics = result["long_direction_metrics"]
    predictions = result["predictions"]
    feature_selection = result.get("feature_selection", pd.DataFrame())
    best_model = None
    if not regression_metrics.empty and "rmse" in regression_metrics.columns:
        best_model = (
            regression_metrics.sort_values("rmse", ascending=True).iloc[0]["model"]
        )

    metadata = {
        "tier": stock_close_pecnet_model_spec.get("tier_name", "tier1"),
        "train_rows": result["train_rows"],
        "test_rows": result["test_rows"],
        "best_model": best_model,
        "regression_metric_rows": len(regression_metrics),
        "long_direction_metric_rows": len(long_direction_metrics),
        "prediction_rows": len(predictions),
        "feature_selection_rows": len(feature_selection),
        "preprocessed_store_rows": result.get("preprocessed_store_rows", 0),
        "models": list(result.get("models", {}).keys()),
    }
    _log_step("train_pecnet_models", **metadata)
    return (
        regression_metrics,
        long_direction_metrics,
        predictions,
        feature_selection,
        metadata,
    )


In [ ]:
# nodes.py
def summarize_training(*metadata_items: dict[str, Any]) -> dict[str, Any]:
    summary = {"sections": list(metadata_items)}
    _log_step("summarize_training", sections=len(metadata_items))
    return summary


In [ ]:
# nodes.py
def summarize_machine_learning(*metadata_items: dict[str, Any]) -> dict[str, Any]:
    summary = {"sections": list(metadata_items)}
    _log_step("summarize_machine_learning", sections=len(metadata_items))
    return summary


## Model Matrix


In [ ]:
# model_matrix.py
def model_tiers() -> tuple[str, ...]:
    return MODEL_TIER_NAMES


In [ ]:
# model_matrix.py
def _dataset(tier_name: str, name: str) -> str:
    return f"stock_close_{tier_name}_{name}"


In [ ]:
# model_matrix.py
def tier_machine_learning_nodes(
    tier_name: str,
    *,
    wait_for_feature_publish: bool = False,
) -> list:
    load_inputs = (
        ["run_config", "model_feature_publish_metadata"]
        if wait_for_feature_publish
        else "run_config"
    )
    load_func = (
        load_model_training_dataset_after_feature_publish
        if wait_for_feature_publish
        else load_model_training_dataset
    )

    return [
        node(
            func=partial(load_func, tier_name=tier_name),
            inputs=load_inputs,
            outputs=[
                _dataset(tier_name, "training_dataset"),
                _dataset(tier_name, "training_dataset_metadata"),
            ],
            name=f"load_{tier_name}_training_dataset",
        ),
        node(
            func=partial(train_test_split_for_tier, tier_name=tier_name),
            inputs=[
                _dataset(tier_name, "training_dataset"),
                "run_config",
            ],
            outputs=[
                _dataset(tier_name, "train_test_split"),
                _dataset(tier_name, "train_test_split_metadata"),
            ],
            name=f"{tier_name}_train_test_split",
        ),
        *_mlforecast_nodes(tier_name),
        *_statsforecast_nodes(tier_name),
        *_pecnet_nodes(tier_name),
    ]


In [ ]:
# model_matrix.py
def model_matrix_nodes(*, wait_for_feature_publish: bool = False) -> list:
    nodes = []
    for tier_name in model_tiers():
        nodes.extend(
            tier_machine_learning_nodes(
                tier_name,
                wait_for_feature_publish=wait_for_feature_publish,
            )
        )
    return nodes


In [ ]:
# model_matrix.py
def model_matrix_summary_inputs() -> list[str]:
    inputs = []
    for tier_name in model_tiers():
        inputs.extend(
            [
                _dataset(tier_name, "training_dataset_metadata"),
                _dataset(tier_name, "train_test_split_metadata"),
                _dataset(tier_name, "mlforecast_training_metadata"),
                _dataset(tier_name, "statsforecast_training_metadata"),
                _dataset(tier_name, "pecnet_training_metadata"),
            ]
        )
    return inputs


In [ ]:
# model_matrix.py
def _mlforecast_nodes(tier_name: str) -> list:
    return [
        node(
            func=partial(build_model_spec, tier_name=tier_name),
            inputs="run_config",
            outputs=_dataset(tier_name, "mlforecast_model_spec"),
            name=f"build_{tier_name}_mlforecast_model_spec",
        ),
        node(
            func=train_models,
            inputs=[
                _dataset(tier_name, "train_test_split"),
                _dataset(tier_name, "mlforecast_model_spec"),
            ],
            outputs=[
                _dataset(tier_name, "mlforecast_regression_metrics"),
                _dataset(tier_name, "mlforecast_long_direction_metrics"),
                _dataset(tier_name, "mlforecast_predictions"),
                _dataset(tier_name, "mlforecast_training_metadata"),
            ],
            name=f"train_{tier_name}_mlforecast_models",
        ),
    ]


In [ ]:
# model_matrix.py
def _statsforecast_nodes(tier_name: str) -> list:
    return [
        node(
            func=partial(build_statsforecast_model_spec_for_tier, tier_name=tier_name),
            inputs="run_config",
            outputs=_dataset(tier_name, "statsforecast_model_spec"),
            name=f"build_{tier_name}_statsforecast_model_spec",
        ),
        node(
            func=train_statsforecast_models,
            inputs=[
                _dataset(tier_name, "train_test_split"),
                _dataset(tier_name, "statsforecast_model_spec"),
            ],
            outputs=[
                _dataset(tier_name, "statsforecast_regression_metrics"),
                _dataset(tier_name, "statsforecast_long_direction_metrics"),
                _dataset(tier_name, "statsforecast_predictions"),
                _dataset(tier_name, "statsforecast_training_metadata"),
            ],
            name=f"train_{tier_name}_statsforecast_models",
        ),
    ]


In [ ]:
def _pecnet_nodes(tier_name: str) -> list:
    return [
        node(
            func=partial(build_pecnet_model_spec_for_tier, tier_name=tier_name),
            inputs="run_config",
            outputs=_dataset(tier_name, "pecnet_model_spec"),
            name=f"build_{tier_name}_pecnet_model_spec",
        ),
        node(
            func=train_pecnet_models,
            inputs=[
                _dataset(tier_name, "train_test_split"),
                _dataset(tier_name, "pecnet_model_spec"),
            ],
            outputs=[
                _dataset(tier_name, "pecnet_regression_metrics"),
                _dataset(tier_name, "pecnet_long_direction_metrics"),
                _dataset(tier_name, "pecnet_predictions"),
                _dataset(tier_name, "pecnet_feature_selection"),
                _dataset(tier_name, "pecnet_training_metadata"),
            ],
            name=f"train_{tier_name}_pecnet_models",
        ),
    ]


## Workflow


In [ ]:
# Notebook workflow helper.
def run_tier(tier_name: str) -> dict:
    training_dataset, training_dataset_metadata = load_model_training_dataset(
        run_config,
        tier_name=tier_name,
    )
    train_test_split, train_test_split_metadata = train_test_split_for_tier(
        training_dataset,
        run_config,
        tier_name=tier_name,
    )

    mlforecast_spec = build_model_spec(run_config, tier_name=tier_name)
    mlforecast_outputs = train_models(train_test_split, mlforecast_spec)

    statsforecast_spec = build_statsforecast_model_spec_for_tier(
        run_config,
        tier_name=tier_name,
    )
    statsforecast_outputs = train_statsforecast_models(
        train_test_split,
        statsforecast_spec,
    )

    pecnet_spec = build_pecnet_model_spec_for_tier(run_config, tier_name=tier_name)
    pecnet_outputs = train_pecnet_models(train_test_split, pecnet_spec)

    return {
        "training_dataset": training_dataset,
        "training_dataset_metadata": training_dataset_metadata,
        "train_test_split": train_test_split,
        "train_test_split_metadata": train_test_split_metadata,
        "mlforecast": {
            "regression": mlforecast_outputs[0],
            "long_direction": mlforecast_outputs[1],
            "predictions": mlforecast_outputs[2],
            "metadata": mlforecast_outputs[3],
        },
        "statsforecast": {
            "regression": statsforecast_outputs[0],
            "long_direction": statsforecast_outputs[1],
            "predictions": statsforecast_outputs[2],
            "metadata": statsforecast_outputs[3],
        },
        "pecnet": {
            "regression": pecnet_outputs[0],
            "long_direction": pecnet_outputs[1],
            "predictions": pecnet_outputs[2],
            "metadata": pecnet_outputs[3],
        },
    }


In [ ]:
# Notebook metric helper.
def collect_metric_tables(results: dict, metric_name: str) -> pd.DataFrame:
    frames = []
    for tier_name, tier_results in results.items():
        for framework_name in ("mlforecast", "statsforecast", "pecnet"):
            frame = tier_results[framework_name][metric_name]
            if frame is None or frame.empty:
                continue
            frames.append(frame.assign(tier=tier_name, framework=framework_name))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


## Runtime Constants


In [ ]:
# Runtime paths and module-level constants live in this one cell.
PROJECT_ROOT = Path.cwd()
WORKSPACE_PROJECT_ROOT = PROJECT_ROOT
if not (WORKSPACE_PROJECT_ROOT / "kedro_project").exists() and (WORKSPACE_PROJECT_ROOT.parent / "kedro_project").exists():
    WORKSPACE_PROJECT_ROOT = WORKSPACE_PROJECT_ROOT.parent
KEDRO_PROJECT = Path("/opt/kedro_project") if Path("/opt/kedro_project").exists() else WORKSPACE_PROJECT_ROOT / "kedro_project"
FEATURE_REPO_SOURCE = Path("/opt/feature_repo") if Path("/opt/feature_repo").exists() else WORKSPACE_PROJECT_ROOT / "feature_repo"
PECNETFRAMEWORK_SOURCE = Path("/opt/pecnetframework") if Path("/opt/pecnetframework").exists() else WORKSPACE_PROJECT_ROOT / "pecnetframework"
STOCK_CLOSE_SOURCE = KEDRO_PROJECT / "src" / "mlops_kedro" / "pipelines" / "stock_close_training"
os.environ.setdefault("KEDRO_PROJECT_DIR", str(KEDRO_PROJECT))
os.environ.setdefault("FEATURE_REPO_DIR", str(FEATURE_REPO_SOURCE))
os.environ.setdefault("PECNETFRAMEWORK_PATH", str(PECNETFRAMEWORK_SOURCE))
plt.style.use("seaborn-v0_8")

# features/feature_sets.py
_COLUMNS = _column_config()

# features/feature_sets.py
ENTITY_COLUMNS = _COLUMNS["entity"]

# features/feature_sets.py
PRICE_COLUMNS = _COLUMNS["price"]

# features/feature_sets.py
ANALYTICS_CALENDAR_COLUMNS = _COLUMNS["analytics_calendar"]

# features/feature_sets.py
TARGET_COLUMNS = _COLUMNS["target"]

# features/feature_sets.py
TIER_1_FEATURE_COLUMNS = _COLUMNS["tier_1_features"]

# features/feature_sets.py
FOURIER_TIME_ENCODING_COLUMNS = _COLUMNS["fourier_time_encoding"]

# features/feature_sets.py
MODEL_TIME_FEATURE_COLUMNS = _COLUMNS["model_time_features"]

# features/feature_sets.py
TIME_FEATURE_COLUMNS = MODEL_TIME_FEATURE_COLUMNS

# features/feature_sets.py
TIER_2_FEATURE_COLUMNS = _COLUMNS["tier_2_features"]

# features/feature_sets.py
TIER_3_FEATURE_COLUMNS = _COLUMNS["tier_3_features"]

# features/feature_sets.py
MODEL_TIER_FEATURE_COLUMNS = {
    "tier1": TIER_1_FEATURE_COLUMNS,
    "tier2": TIER_2_FEATURE_COLUMNS,
    "tier3": TIER_3_FEATURE_COLUMNS,
}

# features/feature_sets.py
MODEL_TIER_NAMES = tuple(MODEL_TIER_FEATURE_COLUMNS.keys())

# features/feature_sets.py
INDICATOR_COLUMNS = _COLUMNS["indicator"]

# features/feature_sets.py
CONDITION_COLUMNS = _COLUMNS["condition"]

# features/feature_sets.py
STRATEGY_LABEL_COLUMNS = _COLUMNS["strategy_label"]

# features/feature_sets.py
OUTPUT_AUDIT_COLUMNS = _COLUMNS["output_audit"]

# features/feature_sets.py
INDICATOR_FEATURE_COLUMNS = _COLUMNS["indicator_features"]

# features/feature_sets.py
CONVENTIONAL_GAP_TRADING_COLUMNS = _COLUMNS["conventional_gap_trading"]

# features/feature_sets.py
STRATEGY_FEATURE_COLUMNS = CONVENTIONAL_GAP_TRADING_COLUMNS

# features/feature_sets.py
FEATURE_COLUMNS = INDICATOR_FEATURE_COLUMNS

# features/feature_sets.py
CLOSE_MODEL_TIME_FEATURE_COLUMNS = _COLUMNS["close_model_time_features"]

# features/feature_sets.py
CLOSE_MODEL_DATASET_COLUMNS = _COLUMNS["close_model_dataset"]

# features/feature_sets.py
FEAST_ENTITY_COLUMNS = [
    *ENTITY_COLUMNS,
    *OUTPUT_AUDIT_COLUMNS,
]

# features/feature_sets.py
FEAST_OFFLINE_COLUMNS = [
    *FEAST_ENTITY_COLUMNS,
    *TIER_2_FEATURE_COLUMNS,
]

# ml/metrics.py
BASE_COLUMNS = {
    "unique_id",
    "ds",
    "y",
    "previous_actual_close",
    "actual_long",
}

# ml/common.py
os.environ.setdefault("MLFLOW_HTTP_REQUEST_TIMEOUT", "60")

# ml/common.py
os.environ.setdefault("MLFLOW_HTTP_REQUEST_MAX_RETRIES", "1")

# ml/common.py
os.environ.setdefault("MLFLOW_HTTP_REQUEST_BACKOFF_FACTOR", "1")

# ml/common.py
DEFAULT_MLFLOW_TRACKING_URI = "http://host.docker.internal:5001"

# ml/common.py
DEFAULT_MLFLOW_EXPERIMENT_NAME = "stock_close_training"

# ml/common.py
DEFAULT_MLFLOW_REQUEST_TIMEOUT = "60"

# ml/common.py
DEFAULT_MLFLOW_REQUEST_MAX_RETRIES = "1"

# ml/common.py
DEFAULT_MLFLOW_REQUEST_BACKOFF_FACTOR = "1"

# ml/common.py
DEFAULT_MLFLOW_TIER_EXPERIMENT_PREFIX = "stock_close"

# ml/mlforecast_training.py
os.environ.setdefault("MLFLOW_HTTP_REQUEST_TIMEOUT", "60")

# ml/mlforecast_training.py
os.environ.setdefault("MLFLOW_HTTP_REQUEST_MAX_RETRIES", "1")

# ml/mlforecast_training.py
os.environ.setdefault("MLFLOW_HTTP_REQUEST_BACKOFF_FACTOR", "1")

# ml/mlforecast_training.py
LOGGER = logging.getLogger(__name__)

# ml/stats_models.py
SUPPORTED_STATSFORECAST_MODELS = {
    "AutoARIMA",
    "AutoCES",
    "AutoETS",
    "AutoMFLES",
    "AutoRegressive",
    "AutoTBATS",
    "AutoTheta",
}

# ml/stats_training.py
LOGGER = logging.getLogger(__name__)

# feature_repo/stock_features.py
TIER_1_FEATURES = [
    "prev_open",
    "prev_high",
    "prev_low",
    "prev_volume",
]

# feature_repo/stock_features.py
TIER_2_TIME_FEATURES = [
    "calendar_gap_days",
    "month_sin_1",
    "month_cos_1",
    "month_sin_2",
    "month_cos_2",
    "day_sin_1",
    "day_cos_1",
    "day_sin_2",
    "day_cos_2",
    "day_of_year_sin_1",
    "day_of_year_cos_1",
    "day_of_year_sin_2",
    "day_of_year_cos_2",
]

# feature_repo/stock_features.py
TIER_2_FEATURES = [
    *TIER_1_FEATURES,
    *TIER_2_TIME_FEATURES,
]

# feature_repo/stock_features.py
TIER_3_FEATURES = TIER_2_FEATURES

# feature_repo/stock_features.py
ticker = Entity(name="ticker", join_keys=["symbol"])

# feature_repo/stock_features.py
stock_model_source = PostgreSQLSource(
    name="stock_model_features_source",
    query="SELECT * FROM feature_store.stock_model_features",
    timestamp_field="date",
    created_timestamp_column="created_timestamp",
)

# feature_repo/stock_features.py
stock_model_features_view = FeatureView(
    name="stock_model_features",
    entities=[ticker],
    ttl=timedelta(days=3650),
    schema=[
        Field(name="prev_open", dtype=Float64),
        Field(name="prev_high", dtype=Float64),
        Field(name="prev_low", dtype=Float64),
        Field(name="prev_volume", dtype=Float64),
        Field(name="calendar_gap_days", dtype=Int64),
        Field(name="month_sin_1", dtype=Float64),
        Field(name="month_cos_1", dtype=Float64),
        Field(name="month_sin_2", dtype=Float64),
        Field(name="month_cos_2", dtype=Float64),
        Field(name="day_sin_1", dtype=Float64),
        Field(name="day_cos_1", dtype=Float64),
        Field(name="day_sin_2", dtype=Float64),
        Field(name="day_cos_2", dtype=Float64),
        Field(name="day_of_year_sin_1", dtype=Float64),
        Field(name="day_of_year_cos_1", dtype=Float64),
        Field(name="day_of_year_sin_2", dtype=Float64),
        Field(name="day_of_year_cos_2", dtype=Float64),
    ],
    online=True,
    source=stock_model_source,
    tags={"layer": "feature_serving", "team": "dataops_mlops"},
)

# feature_repo/stock_features.py
stock_model_tier_1_feature_service = FeatureService(
    name="stock_model_tier_1_features_v1",
    features=[stock_model_features_view[TIER_1_FEATURES]],
)

# feature_repo/stock_features.py
stock_model_tier_2_feature_service = FeatureService(
    name="stock_model_tier_2_features_v1",
    features=[stock_model_features_view[TIER_2_FEATURES]],
)

# feature_repo/stock_features.py
stock_model_tier_3_feature_service = FeatureService(
    name="stock_model_tier_3_features_v1",
    features=[stock_model_features_view[TIER_3_FEATURES]],
)

# feature_repo/stock_features.py
stock_feature_row = Entity(name="stock_feature_row", join_keys=["feature_key"])

# feature_repo/stock_features.py
stock_model_tier2_dataset_source = PostgreSQLSource(
    name="stock_model_tier2_dataset_source",
    query="""
        SELECT
            *,
            to_char(
                "date" AT TIME ZONE 'UTC',
                'YYYY-MM-DD"T"HH24:MI:SS.US"Z"'
            ) AS date_key,
            symbol || '|' || to_char(
                "date" AT TIME ZONE 'UTC',
                'YYYY-MM-DD"T"HH24:MI:SS.US"Z"'
            ) AS feature_key
        FROM feature_store.stock_model_features
    """,
    timestamp_field="date",
    created_timestamp_column="created_timestamp",
)

# feature_repo/stock_features.py
stock_model_tier2_dataset_view = FeatureView(
    name="stock_model_tier2_dataset",
    entities=[stock_feature_row],
    ttl=timedelta(days=3650),
    schema=[
        Field(name="prev_open", dtype=Float64),
        Field(name="prev_high", dtype=Float64),
        Field(name="prev_low", dtype=Float64),
        Field(name="prev_volume", dtype=Float64),
        Field(name="calendar_gap_days", dtype=Int64),
        Field(name="month_sin_1", dtype=Float64),
        Field(name="month_cos_1", dtype=Float64),
        Field(name="month_sin_2", dtype=Float64),
        Field(name="month_cos_2", dtype=Float64),
        Field(name="day_sin_1", dtype=Float64),
        Field(name="day_cos_1", dtype=Float64),
        Field(name="day_sin_2", dtype=Float64),
        Field(name="day_cos_2", dtype=Float64),
        Field(name="day_of_year_sin_1", dtype=Float64),
        Field(name="day_of_year_cos_1", dtype=Float64),
        Field(name="day_of_year_sin_2", dtype=Float64),
        Field(name="day_of_year_cos_2", dtype=Float64),
    ],
    online=True,
    source=stock_model_tier2_dataset_source,
    tags={"layer": "model_training", "team": "dataops_mlops"},
)

# feature_repo/stock_features.py
stock_model_tier2_dataset_service = FeatureService(
    name="stock_model_tier2_dataset_v1",
    features=[stock_model_tier2_dataset_view[TIER_2_FEATURES]],
)

# feature_repo/stock_features.py
stock_series = Entity(name="stock_series", join_keys=["series_key"])

# feature_repo/stock_features.py
stock_close_model_dataset_source = PostgreSQLSource(
    name="stock_close_model_dataset_source",
    query="""
        SELECT
            *,
            to_char(
                ds AT TIME ZONE 'UTC',
                'YYYY-MM-DD"T"HH24:MI:SS.US"Z"'
            ) AS ds_key,
            unique_id || '|' || to_char(
                ds AT TIME ZONE 'UTC',
                'YYYY-MM-DD"T"HH24:MI:SS.US"Z"'
            ) AS series_key
        FROM feature_store.stock_close_model_dataset
    """,
    timestamp_field="ds",
    created_timestamp_column="created_timestamp",
)

# feature_repo/stock_features.py
stock_close_model_dataset_view = FeatureView(
    name="stock_close_model_dataset",
    entities=[stock_series],
    ttl=timedelta(days=3650),
    schema=[
        Field(name="y", dtype=Float64),
        Field(name="month_sin_1", dtype=Float64),
        Field(name="month_cos_1", dtype=Float64),
        Field(name="day_sin_1", dtype=Float64),
        Field(name="day_cos_1", dtype=Float64),
        Field(name="day_of_year_sin_1", dtype=Float64),
        Field(name="day_of_year_cos_1", dtype=Float64),
    ],
    online=True,
    source=stock_close_model_dataset_source,
    tags={"layer": "model_training", "team": "dataops_mlops"},
)

# feature_repo/stock_features.py
stock_close_model_dataset_service = FeatureService(
    name="stock_close_model_dataset_v1",
    features=[
        stock_close_model_dataset_view[
            [
                "y",
                "month_sin_1",
                "month_cos_1",
                "day_sin_1",
                "day_cos_1",
                "day_of_year_sin_1",
                "day_of_year_cos_1",
            ]
        ]
    ],
)

# feature_repo/stock_features.py
pecnet_preprocessed_row = Entity(
    name="pecnet_preprocessed_row",
    join_keys=["row_key"],
)

# feature_repo/stock_features.py
pecnet_preprocessed_training_source = PostgreSQLSource(
    name="pecnet_preprocessed_training_source",
    query="SELECT * FROM feature_store.pecnet_preprocessed_training_data",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

# feature_repo/stock_features.py
pecnet_preprocessed_training_view = FeatureView(
    name="pecnet_preprocessed_training_data",
    entities=[pecnet_preprocessed_row],
    ttl=timedelta(days=3650),
    schema=[
        Field(name="value", dtype=Float64),
        Field(name="target_y", dtype=Float64),
        Field(name="sample_index", dtype=Int64),
        Field(name="step_index", dtype=Int64),
        Field(name="variable_index", dtype=Int64),
        Field(name="split_index", dtype=Int64),
    ],
    online=True,
    source=pecnet_preprocessed_training_source,
    tags={"layer": "model_training", "model": "pecnet", "team": "dataops_mlops"},
)

# feature_repo/stock_features.py
pecnet_preprocessed_training_service = FeatureService(
    name="pecnet_preprocessed_training_data_v1",
    features=[
        pecnet_preprocessed_training_view[
            [
                "value",
                "target_y",
                "sample_index",
                "step_index",
                "variable_index",
                "split_index",
            ]
        ]
    ],
)

# serving/feast_store.py
FEATURE_REPO_DIR = Path(
    os.getenv("FEATURE_REPO_DIR")
    or str(globals().get("FEATURE_REPO_SOURCE", Path("/opt/feature_repo")))
).resolve()

# serving/feast_store.py
TIMESCALE_TABLE = "feature_store.stock_model_features"

# serving/feast_store.py
TIMESCALE_CLOSE_MODEL_DATASET_TABLE = "feature_store.stock_close_model_dataset"

# serving/feast_store.py
TIMESCALE_CONVENTIONAL_GAP_TRADING_TABLE = "feature_store.conventional_gap_trading"

# serving/feast_store.py
TIMESCALE_PECNET_PREPROCESSED_TABLE = (
    "feature_store.pecnet_preprocessed_training_data"
)

# serving/feast_store.py
TIMESCALE_WRITE_BATCH_SIZE = int(os.getenv("TIMESCALE_WRITE_BATCH_SIZE", "500"))

# serving/feast_store.py
TIMESCALE_DAILY_FILL_FREQ = os.getenv("TIMESCALE_DAILY_FILL_FREQ", "B")

# serving/feast_store.py
PECNET_PREPROCESSED_COLUMNS = [
    "row_key",
    "tier",
    "symbol",
    "event_timestamp",
    "split",
    "split_index",
    "variable_name",
    "variable_index",
    "sample_index",
    "step_index",
    "value",
    "target_y",
    "created_timestamp",
]

# serving/feast_store.py
PECNET_PREPROCESSED_FEAST_COLUMNS = [
    "row_key",
    "event_timestamp",
    "created_timestamp",
    "value",
    "target_y",
    "sample_index",
    "step_index",
    "variable_index",
    "split_index",
]

# ml/pecnet_training.py
LOGGER = logging.getLogger(__name__)


## Notebook Helpers


In [ ]:
# Notebook helper: YAML loader used by workflow cells
def load_yaml(path: str | Path) -> dict[str, Any]:
    path = Path(path).expanduser()
    try:
        with path.open() as file:
            return yaml.safe_load(file) or {}
    except OSError as exc:
        if getattr(exc, "errno", None) == errno.EDEADLK:
            opt_candidate = Path("/opt/kedro_project") / path.name
            if path.name.startswith("parameters_"):
                opt_candidate = Path("/opt/kedro_project/conf/base") / path.name
            if opt_candidate.exists():
                with opt_candidate.open() as file:
                    return yaml.safe_load(file) or {}
        raise


## Run Configuration


In [ ]:
data_preprocessing_params = load_yaml(
    KEDRO_PROJECT / "conf" / "base" / "parameters_data_preprocessing.yml"
)["stock_close_data_preprocessing"]
machine_learning_params = load_yaml(
    KEDRO_PROJECT / "conf" / "base" / "parameters_machine_learning.yml"
)["stock_close_machine_learning"]

run_config = start_config(data_preprocessing_params, machine_learning_params)
TIERS = tuple(tier for tier in model_tiers() if tier in run_config["model_tier_feature_columns"])
TIERS


In [ ]:
results = {}
for tier_name in TIERS:
    results[tier_name] = run_tier(tier_name)

results.keys()


## Metric Tables


In [ ]:
regression_metrics = collect_metric_tables(results, "regression")
long_direction_metrics = collect_metric_tables(results, "long_direction")

display(regression_metrics)
display(long_direction_metrics)


## Regression Comparison


In [ ]:
if not regression_metrics.empty and {"tier", "framework", "model", "rmse"}.issubset(regression_metrics.columns):
    plot_df = regression_metrics.copy()
    plot_df["label"] = plot_df["tier"] + " / " + plot_df["framework"] + " / " + plot_df["model"].astype(str)
    plot_df = plot_df.sort_values("rmse")
    ax = plot_df.plot.barh(x="label", y="rmse", figsize=(12, 8), legend=False)
    ax.set_title("RMSE by tier, framework and model")
    ax.set_xlabel("rmse")
    ax.set_ylabel("")
    plt.tight_layout()


## Long-Only Directional Accuracy


In [ ]:
if not long_direction_metrics.empty and {"tier", "framework", "model", "long_accuracy"}.issubset(long_direction_metrics.columns):
    plot_df = long_direction_metrics[
        long_direction_metrics["model"].astype(str).ne("previous_actual_close")
    ].copy()
    plot_df["label"] = plot_df["tier"] + " / " + plot_df["framework"] + " / " + plot_df["model"].astype(str)
    plot_df = plot_df.sort_values("long_accuracy")
    ax = plot_df.plot.barh(x="label", y="long_accuracy", figsize=(12, 8), legend=False)
    ax.set_title("Long-only directional accuracy")
    ax.set_xlabel("accuracy")
    ax.set_ylabel("")
    plt.tight_layout()
